# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 3/12** (CPU, internet off). **1778 series**, ~10.6 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part3.zip`
- `manifest_rgb25d_320_part3.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 3
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260","series":"1.2.826.0.1.3680043.8.498.12343110195036213483454091715412333772","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764","series":"1.2.826.0.1.3680043.8.498.12899608222010739549162228949977121036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339","series":"1.2.826.0.1.3680043.8.498.67294291903707258636363422483846920874","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606","series":"1.2.826.0.1.3680043.8.498.36575741184530379556865901754021617448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394","series":"1.2.826.0.1.3680043.8.498.22350941940168581233659964138249514718","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.11788164428905677775308855476550714595","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866","series":"1.2.826.0.1.3680043.8.498.33788855095027519865615604807327323172","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269","series":"1.2.826.0.1.3680043.8.498.58849834278018158596588761481850832613","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736","series":"1.2.826.0.1.3680043.8.498.11179656284155940848820194004757335696","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941","series":"1.2.826.0.1.3680043.8.498.62528033445367515360713711873528975698","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365","series":"1.2.826.0.1.3680043.8.498.18366537086670924043033803636085833012","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678","series":"1.2.826.0.1.3680043.8.498.10351152397204164344416168902069476285","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863","series":"1.2.826.0.1.3680043.8.498.13936472283003825905136712381314777313","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659","series":"1.2.826.0.1.3680043.8.498.45860069773213535583606324988544204643","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100","series":"1.2.826.0.1.3680043.8.498.12098721286482739004113481468727516358","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339","series":"1.2.826.0.1.3680043.8.498.63693852990818045309124115231693705598","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833","series":"1.2.826.0.1.3680043.8.498.23018906617964676055055758476109628559","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166","series":"1.2.826.0.1.3680043.8.498.13405009385902643892114787851848890052","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104","series":"1.2.826.0.1.3680043.8.498.34250391165634005487105222319632670933","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314","series":"1.2.826.0.1.3680043.8.498.57044628117746084564188065037591878208","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586","series":"1.2.826.0.1.3680043.8.498.14991345816284084660769375156189038685","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019","series":"1.2.826.0.1.3680043.8.498.37237025409410099950666378948790724886","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205","series":"1.2.826.0.1.3680043.8.498.12635219996711974925045224418830382646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484","series":"1.2.826.0.1.3680043.8.498.84254179007469624300482931026762442976","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478","series":"1.2.826.0.1.3680043.8.498.61647627627816288079198049926960858673","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10165669527316708372190215034240922433","series":"1.2.826.0.1.3680043.8.498.21444945880760356085615309481698551742","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787","series":"1.2.826.0.1.3680043.8.498.19396300033493401328740387893770546496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292","series":"1.2.826.0.1.3680043.8.498.58587492011390814873558377267809213645","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507","series":"1.2.826.0.1.3680043.8.498.32512483619641874454128441636792579533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773","series":"1.2.826.0.1.3680043.8.498.14742042201485078869478491511180615133","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775","series":"1.2.826.0.1.3680043.8.498.10967877394372926199437908171011016345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585","series":"1.2.826.0.1.3680043.8.498.74051993175716365349762044490292995240","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375","series":"1.2.826.0.1.3680043.8.498.50240933564321985896439997614989804804","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.24597592709320826129157256964972888672","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10240166440044087554182120832488098189","series":"1.2.826.0.1.3680043.8.498.83812899145092855711745333206516412902","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307","series":"1.2.826.0.1.3680043.8.498.11653397496259754983550419128684121716","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531","series":"1.2.826.0.1.3680043.8.498.74524777621297243054896376044173013085","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.61195867264290100563921711711852433937","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903","series":"1.2.826.0.1.3680043.8.498.26094787216486195409494347514320460157","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.96225637366864847874681571967453119985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029","series":"1.2.826.0.1.3680043.8.498.49415030731924373550448811716001025587","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639","series":"1.2.826.0.1.3680043.8.498.10641081243353851892021905881692814843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669","series":"1.2.826.0.1.3680043.8.498.79192397872238774975064518353237300355","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097","series":"1.2.826.0.1.3680043.8.498.11809237605835830560992785754681212833","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052","series":"1.2.826.0.1.3680043.8.498.10903465965639594231540335074980096554","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","series":"1.2.826.0.1.3680043.8.498.56798972099993394367643876357351680742","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846","series":"1.2.826.0.1.3680043.8.498.62900515279367641290195588302084547379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321702681363532511953946967334431952","series":"1.2.826.0.1.3680043.8.498.55634493664532706203327179965330282989","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10322853084122920408377349136830993432","series":"1.2.826.0.1.3680043.8.498.17697711597859086507295300312348615087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687","series":"1.2.826.0.1.3680043.8.498.28843655461443652834287897281876397783","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10336300544286006569730381727202519821","series":"1.2.826.0.1.3680043.8.498.62928058878742677290148933101280575226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306","series":"1.2.826.0.1.3680043.8.498.12296411592288272155821437847678584269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385","series":"1.2.826.0.1.3680043.8.498.10005047919793760894599773106036876356","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355","series":"1.2.826.0.1.3680043.8.498.31573346262550295678201711757418061957","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10370612191212456931506340727148605247","series":"1.2.826.0.1.3680043.8.498.81993872744318743306902832957160693563","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10378535286770237477615181844634903700","series":"1.2.826.0.1.3680043.8.498.67828784545173031060975957383653064278","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500","series":"1.2.826.0.1.3680043.8.498.10843417049434653564450890800518831128","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432","series":"1.2.826.0.1.3680043.8.498.12179838061226441971400872938504960368","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933","series":"1.2.826.0.1.3680043.8.498.20559399698413776656974357356837441906","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334","series":"1.2.826.0.1.3680043.8.498.10262105074293839185767303087060685562","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434","series":"1.2.826.0.1.3680043.8.498.91951761512357579699262111665483518351","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715","series":"1.2.826.0.1.3680043.8.498.51868617929771884999773457192466193614","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10419993051271996183722192755371692029","series":"1.2.826.0.1.3680043.8.498.11771752034240094063118985489471941832","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10423586531262405323219596291234698626","series":"1.2.826.0.1.3680043.8.498.10722483446455229968737875672916311986","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056","series":"1.2.826.0.1.3680043.8.498.65249626378501201881028459892404902225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10434135978927658622039675788413759069","series":"1.2.826.0.1.3680043.8.498.12628652805931734067386926576225932362","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409","series":"1.2.826.0.1.3680043.8.498.13372515709977940542846289958094038890","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397","series":"1.2.826.0.1.3680043.8.498.92522052617543368794379920033825917007","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.10617525032286442761923502016480664304","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695","series":"1.2.826.0.1.3680043.8.498.56422561960613581217952138778230724736","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546","series":"1.2.826.0.1.3680043.8.498.12865605300249850040931642953040243874","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584","series":"1.2.826.0.1.3680043.8.498.12529748410625327431924424654201427209","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117","series":"1.2.826.0.1.3680043.8.498.22830479045278174488866068260623943646","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454","series":"1.2.826.0.1.3680043.8.498.19704356800987589610559423149057051480","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522","series":"1.2.826.0.1.3680043.8.498.80951243654806465601559003669953177710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10521372086358356219840179458708706015","series":"1.2.826.0.1.3680043.8.498.39615291577880315881501285086759216640","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018","series":"1.2.826.0.1.3680043.8.498.26296461508151029163309274666416127090","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739","series":"1.2.826.0.1.3680043.8.498.88949277510163367479649532465753343087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327","series":"1.2.826.0.1.3680043.8.498.11001958803754225074389902755716478577","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573","series":"1.2.826.0.1.3680043.8.498.79567238824290935428384132163455226043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541029960718591409318399557079055927","series":"1.2.826.0.1.3680043.8.498.92679627845520691792217237021162883135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251","series":"1.2.826.0.1.3680043.8.498.49560781421342916443519696772141175722","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736","series":"1.2.826.0.1.3680043.8.498.68900108284635015233771931587692405280","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109","series":"1.2.826.0.1.3680043.8.498.42533775174102881956433036981345758571","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169","series":"1.2.826.0.1.3680043.8.498.34032569739843351865030993307195163534","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10589889055723532336864750992142411145","series":"1.2.826.0.1.3680043.8.498.46635861441006784875157982795399826512","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10591507262178959553431147886647585916","series":"1.2.826.0.1.3680043.8.498.21592605516704061897858405038361725817","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137","series":"1.2.826.0.1.3680043.8.498.10683654154977180409193787516365396378","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146","series":"1.2.826.0.1.3680043.8.498.46635940681606147009204301777014526298","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241","series":"1.2.826.0.1.3680043.8.498.56822829637909681029718998994047504953","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403","series":"1.2.826.0.1.3680043.8.498.63727968281770767444988085873821315427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430","series":"1.2.826.0.1.3680043.8.498.13116655939906914102412938793438358404","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10626411974489938664325062396760010912","series":"1.2.826.0.1.3680043.8.498.80645852125773796384711164478522678225","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477","series":"1.2.826.0.1.3680043.8.498.31101863956989536709890546027001820215","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066","series":"1.2.826.0.1.3680043.8.498.79534244788628168753834278038944217620","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374","series":"1.2.826.0.1.3680043.8.498.21785088381805751892490720966845719763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658","series":"1.2.826.0.1.3680043.8.498.11013503879363481829718099650262938749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831","series":"1.2.826.0.1.3680043.8.498.75296849418011774495226872505344244675","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082","series":"1.2.826.0.1.3680043.8.498.53779798159446473712456225009142994603","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731","series":"1.2.826.0.1.3680043.8.498.10428193422126181383053153342738115224","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331","series":"1.2.826.0.1.3680043.8.498.19987097303914992766164853215772846169","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592","series":"1.2.826.0.1.3680043.8.498.41840670210316247998213899104451438617","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10699348333280906956391277892869999716","series":"1.2.826.0.1.3680043.8.498.38682049893333823113099736637625051102","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811","series":"1.2.826.0.1.3680043.8.498.37699883166730297899096656738917451104","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10724604223190030113012801427752979879","series":"1.2.826.0.1.3680043.8.498.87499889441181562196305777674976761061","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557","series":"1.2.826.0.1.3680043.8.498.53550886318339095631934159953667654712","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10736346409444614168571281812134757732","series":"1.2.826.0.1.3680043.8.498.12603138509227972416136242592110244995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519","series":"1.2.826.0.1.3680043.8.498.67206207672329386899935646323662348206","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10750145216793948201921667945805555093","series":"1.2.826.0.1.3680043.8.498.11567578038495164598097361967838330631","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327","series":"1.2.826.0.1.3680043.8.498.59789466710140555805160608003879597301","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.12414368360314712251919576458639408508","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.11126851762792686626747289910626409096","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10769126730281890056368854871900324448","series":"1.2.826.0.1.3680043.8.498.74905242071999992362185831251713984776","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555","series":"1.2.826.0.1.3680043.8.498.38386396910843489804587533351832228438","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770","series":"1.2.826.0.1.3680043.8.498.58721682632661588848074976538618213287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077","series":"1.2.826.0.1.3680043.8.498.91843701017719660320425404617635397057","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10812894672739619566834776054635588406","series":"1.2.826.0.1.3680043.8.498.28263126392094756676415281006304937207","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10824420630965162625242888916805538597","series":"1.2.826.0.1.3680043.8.498.26265764259282212122888165424371516517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10835889400414225749339608096119636738","series":"1.2.826.0.1.3680043.8.498.86277781798381038064413895290529785089","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263","series":"1.2.826.0.1.3680043.8.498.21506701195745978123536788550294064807","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650","series":"1.2.826.0.1.3680043.8.498.14174162007767386825128735757135001778","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10851562687641986545442408470139368318","series":"1.2.826.0.1.3680043.8.498.26084376788783968382054780494707921750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766","series":"1.2.826.0.1.3680043.8.498.11062155738430646385275301535196670414","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551","series":"1.2.826.0.1.3680043.8.498.30266494261627905551026428353342440450","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385","series":"1.2.826.0.1.3680043.8.498.10647336162812421823422797117563306403","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10880553082640412137979450100928382021","series":"1.2.826.0.1.3680043.8.498.80118566037581465106578031317627946407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10890699078723972693302029683905114834","series":"1.2.826.0.1.3680043.8.498.99084327872206066550067402621196598033","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10902166579916526176697193295886992412","series":"1.2.826.0.1.3680043.8.498.10891094657236140991659855236518100906","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881","series":"1.2.826.0.1.3680043.8.498.12053486118919659792222878149126286986","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235","series":"1.2.826.0.1.3680043.8.498.50447949078198621390396247897134585491","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10924995713871537837233985073507509566","series":"1.2.826.0.1.3680043.8.498.55487314158239162420729034927691965046","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10930374192038896613513882597332665403","series":"1.2.826.0.1.3680043.8.498.14545499952035780555207771068040156095","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10933003998197427979325590791540457658","series":"1.2.826.0.1.3680043.8.498.45723974658433803179836498909863894947","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768","series":"1.2.826.0.1.3680043.8.498.42531309363032085435419792637210613938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409","series":"1.2.826.0.1.3680043.8.498.36639986044461724921668168249439837501","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514","series":"1.2.826.0.1.3680043.8.498.11725796187393242166326220202445156204","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245","series":"1.2.826.0.1.3680043.8.498.30295791499048372369501139558428569741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528","series":"1.2.826.0.1.3680043.8.498.10917762304353522942663331343716035242","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962","series":"1.2.826.0.1.3680043.8.498.83690178930371005813774190245621000775","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100","series":"1.2.826.0.1.3680043.8.498.46990133807956193604390684745843398099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994899269366718796438389394954844979","series":"1.2.826.0.1.3680043.8.498.52757175269837377453229074372583777458","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10998144118962608870602180672090998054","series":"1.2.826.0.1.3680043.8.498.21666925125173466868504466866097655851","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256","series":"1.2.826.0.1.3680043.8.498.18415514265022206792361682437440100920","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11011061033863592500429481719734134286","series":"1.2.826.0.1.3680043.8.498.74022366317597673224050781012811122857","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088","series":"1.2.826.0.1.3680043.8.498.90892430931741690535455608333671842090","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043543420859531726426521315399258528","series":"1.2.826.0.1.3680043.8.498.10792825035289868891440557269760397759","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11049882927267755769995376553330493298","series":"1.2.826.0.1.3680043.8.498.58154355439815622939542587436050194750","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11052720536194493057619720620119095739","series":"1.2.826.0.1.3680043.8.498.75567391372686069962652971920300538173","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11060834081463491262807709963877480026","series":"1.2.826.0.1.3680043.8.498.12751804616396515154865178948161070577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089","series":"1.2.826.0.1.3680043.8.498.67911581961514266607819483870809797993","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11080651962904815250736183346281427445","series":"1.2.826.0.1.3680043.8.498.10433781525390960133544993407624554814","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11090884548990390279582832205294943635","series":"1.2.826.0.1.3680043.8.498.87844843558328513262968765618402496755","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233","series":"1.2.826.0.1.3680043.8.498.15309294241026255275263161991632901951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427","series":"1.2.826.0.1.3680043.8.498.12428661641561428031510177499938309062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213","series":"1.2.826.0.1.3680043.8.498.61331218641893203362518004354311084381","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11125718814639515292482773418372267821","series":"1.2.826.0.1.3680043.8.498.64560123347492431965793443891711933562","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11128099896284546522122813576483001027","series":"1.2.826.0.1.3680043.8.498.61808734669982649590623048420737073303","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11131210140372132238166068091728877219","series":"1.2.826.0.1.3680043.8.498.31751564370124535591795950564107486369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","series":"1.2.826.0.1.3680043.8.498.11100802624184418021223847670451651947","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960","series":"1.2.826.0.1.3680043.8.498.36436960154373810873908916689136180902","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456","series":"1.2.826.0.1.3680043.8.498.71562747752081100803384284779828729792","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11154238555579687338157479416363307605","series":"1.2.826.0.1.3680043.8.498.90132700516635818292691083231408748173","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11159546221032123495135103648570690656","series":"1.2.826.0.1.3680043.8.498.34601172349533904272121677406920725165","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392","series":"1.2.826.0.1.3680043.8.498.81216815328203788894274604866286169692","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863","series":"1.2.826.0.1.3680043.8.498.77014852110840872909625135283120624780","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131","series":"1.2.826.0.1.3680043.8.498.12258672811607867127738435450432797701","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11207013225631251718312189933025529824","series":"1.2.826.0.1.3680043.8.498.69589676735019735983117127949784257753","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11208663314213048426907645330143931419","series":"1.2.826.0.1.3680043.8.498.10329752736786565551610324709090392574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731","series":"1.2.826.0.1.3680043.8.498.94010225326993046549132576989195636860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627","series":"1.2.826.0.1.3680043.8.498.43785816289831494855680741236068312120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11221318524515817857172161854667955115","series":"1.2.826.0.1.3680043.8.498.82666001696432084414116797305720869547","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11226582925516370364253861721606580867","series":"1.2.826.0.1.3680043.8.498.31202608765349538559685394062266061309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.45016666091171770991519010723812640837","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11249981834802444119817546155435759007","series":"1.2.826.0.1.3680043.8.498.50349184734969313204902337569597640650","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11257869433929994221899994675448655047","series":"1.2.826.0.1.3680043.8.498.54941004602590534222644550753754282516","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11262944211104069231500934582412003273","series":"1.2.826.0.1.3680043.8.498.12561832006870204845598651728874237766","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.12441708157585025854872523785138286688","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.46165743730422054749973702946995549381","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11277278232377185347750793496260885546","series":"1.2.826.0.1.3680043.8.498.56070041243741201820339187537960175809","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282706861378379965012380644125662529","series":"1.2.826.0.1.3680043.8.498.13188932516677186842049309054466957679","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509","series":"1.2.826.0.1.3680043.8.498.83264982894576976680377776635607930114","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.81739280016559050332394885945464238064","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393","series":"1.2.826.0.1.3680043.8.498.13164727315204712699467894629040467681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11312798102268182343541500463824266929","series":"1.2.826.0.1.3680043.8.498.10039299706120855964847841105762943154","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11315871586936178043350206344448851055","series":"1.2.826.0.1.3680043.8.498.43388727232769726098316618712913970291","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712","series":"1.2.826.0.1.3680043.8.498.13201132079462947912734182972220430265","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11337616353935430160375524217870968575","series":"1.2.826.0.1.3680043.8.498.12818069953672323292277208897442694104","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473","series":"1.2.826.0.1.3680043.8.498.10764908852564056138856815005084266226","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11352722814008170661344291337061231555","series":"1.2.826.0.1.3680043.8.498.43793062896885921272768660228349449704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897","series":"1.2.826.0.1.3680043.8.498.10273275087891513440592863286096307301","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.91903627435354016135777336404933338336","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11386074316342852649653530230563922899","series":"1.2.826.0.1.3680043.8.498.85195147751949999678285931064201945501","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11394861774586578112386386342670500749","series":"1.2.826.0.1.3680043.8.498.39216590551819379672243675532955966772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602","series":"1.2.826.0.1.3680043.8.498.12555450896224043267473970337409194095","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512","series":"1.2.826.0.1.3680043.8.498.20247267893827592389292879835063193482","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127","series":"1.2.826.0.1.3680043.8.498.11682797134908455793320529260845747557","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.12668247246532282883837439549474301617","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794","series":"1.2.826.0.1.3680043.8.498.11284590322198448041314910888138296478","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623","series":"1.2.826.0.1.3680043.8.498.12545240630052505223060197904037913588","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11448865770537436004548718998804636234","series":"1.2.826.0.1.3680043.8.498.12803687254991942558907635422861414394","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769","series":"1.2.826.0.1.3680043.8.498.58609126851744856698425476705851755794","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.10950009031711963893953363837521414329","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592","series":"1.2.826.0.1.3680043.8.498.51696731126515840505976196431669730183","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558","series":"1.2.826.0.1.3680043.8.498.17428455312519423561844856305028382999","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11477583622065195478886525904745372063","series":"1.2.826.0.1.3680043.8.498.34906314244292537613145771711831749772","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11489881286208501558797333161632486099","series":"1.2.826.0.1.3680043.8.498.12426161928826971650775878199232288086","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.10853671211430654352514510850554237271","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498505068787084440256890724559761815","series":"1.2.826.0.1.3680043.8.498.37470739625349293073696680428925155237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588","series":"1.2.826.0.1.3680043.8.498.36303855961491708673005160427821546170","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11506856887791286395737465774061395929","series":"1.2.826.0.1.3680043.8.498.64986803803447779055032805486812482195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433","series":"1.2.826.0.1.3680043.8.498.66947318167476697180670328643115853468","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11520508017843738484108845144418061602","series":"1.2.826.0.1.3680043.8.498.26271680367217405301749651444574487411","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983","series":"1.2.826.0.1.3680043.8.498.80437585378057984170148964914568756228","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463","series":"1.2.826.0.1.3680043.8.498.23357245979280982805199693679119023084","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974","series":"1.2.826.0.1.3680043.8.498.43788681565818988307469239050320510069","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935","series":"1.2.826.0.1.3680043.8.498.12942837807178755325452698984524562225","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11551347885357242022697196552822735795","series":"1.2.826.0.1.3680043.8.498.11457115155245525023726804254801073093","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962","series":"1.2.826.0.1.3680043.8.498.96379684941862716805955569469948325675","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425","series":"1.2.826.0.1.3680043.8.498.36879935085970176447820971606899843008","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967","series":"1.2.826.0.1.3680043.8.498.80367116620046594361401129215108207018","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11579571696748051185342754881791715610","series":"1.2.826.0.1.3680043.8.498.33310408022783128646412973281474145741","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11586616391541918134110642641868079265","series":"1.2.826.0.1.3680043.8.498.11775596499462328643441269258376375184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11590096889491409910425456896187293515","series":"1.2.826.0.1.3680043.8.498.57958159116471122325226234815888549683","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789","series":"1.2.826.0.1.3680043.8.498.15664463013254836391314787802652410099","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202","series":"1.2.826.0.1.3680043.8.498.81717862291002129294004843036500723898","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11604138460812393624691789275263090511","series":"1.2.826.0.1.3680043.8.498.12223791147765751096133501221501241147","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11605266731872772432220415835172557961","series":"1.2.826.0.1.3680043.8.498.48672777476739796105883375818515637609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11613637647831801354398273334552399874","series":"1.2.826.0.1.3680043.8.498.80232839691236434379467430789489613339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378","series":"1.2.826.0.1.3680043.8.498.10958828373915910081624857291676385831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612","series":"1.2.826.0.1.3680043.8.498.10964291403679607157037669485702641600","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014","series":"1.2.826.0.1.3680043.8.498.99860828832967173530191638320777493310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11651755040149549966484720727140190853","series":"1.2.826.0.1.3680043.8.498.10985655479406971602314885997770061455","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11654064547593365398103107850228629066","series":"1.2.826.0.1.3680043.8.498.60298992056961218190928129503624287804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493","series":"1.2.826.0.1.3680043.8.498.13211934636969582314745208733007903486","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11674925658184069242322832524274428408","series":"1.2.826.0.1.3680043.8.498.75063918192937466796915287714962438229","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171","series":"1.2.826.0.1.3680043.8.498.49093905640862733123440960170337140007","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11689820638581431730501617718741779835","series":"1.2.826.0.1.3680043.8.498.14297023582134892230230813777307855455","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.57758266410684389014515409351085614067","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11709909941833245598246010905322129256","series":"1.2.826.0.1.3680043.8.498.72262315519171099892555010817789541032","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429","series":"1.2.826.0.1.3680043.8.498.44456811837595069089378687127077492422","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.77518546044142678138425999391875265210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11723286647024326363349453996818420374","series":"1.2.826.0.1.3680043.8.498.46404640525097166232488759105882449606","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11726452645519338341002251323463736083","series":"1.2.826.0.1.3680043.8.498.12142438132071352559312158023716193050","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11752091432368750506312921028559788225","series":"1.2.826.0.1.3680043.8.498.27121009003564950503195859429664607941","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11762851510866726051342964961256680025","series":"1.2.826.0.1.3680043.8.498.10029502290189977905668112003995443939","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756","series":"1.2.826.0.1.3680043.8.498.38661357433276987043969996027193861223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11779387602988880691008088661075117406","series":"1.2.826.0.1.3680043.8.498.11810003266876329999273663660008597895","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663","series":"1.2.826.0.1.3680043.8.498.44817125693505410958840677842536371414","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483","series":"1.2.826.0.1.3680043.8.498.13152395825250279270407514668731607617","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033","series":"1.2.826.0.1.3680043.8.498.83252241597109034586001613706950136062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.12727886398692612565745457715791649472","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758","series":"1.2.826.0.1.3680043.8.498.12310954864680582903733245836188834811","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11807494462283790976935891042637502434","series":"1.2.826.0.1.3680043.8.498.12652200606231849842121225232760031272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108","series":"1.2.826.0.1.3680043.8.498.40956047582708391748396390790140492302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11836929166973549727078374306175992474","series":"1.2.826.0.1.3680043.8.498.21842516232632991865622975426602905738","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604","series":"1.2.826.0.1.3680043.8.498.99189356480890482418477312767560138524","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862","series":"1.2.826.0.1.3680043.8.498.47241939890604341530405125019748105372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11868389442143514705330687746274512632","series":"1.2.826.0.1.3680043.8.498.83328620699513126439598115899372516364","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11878847808072119549790690649256090471","series":"1.2.826.0.1.3680043.8.498.13220774216442023914436268119922121220","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229","series":"1.2.826.0.1.3680043.8.498.79162969256717561964850049656497606406","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794","series":"1.2.826.0.1.3680043.8.498.53001936016165838390822286326847155206","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11902464730418099436837163594625541861","series":"1.2.826.0.1.3680043.8.498.90249526383563833593394098335236390753","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11905397531490569218559534495408776672","series":"1.2.826.0.1.3680043.8.498.12987050833249586134205586019724910921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708","series":"1.2.826.0.1.3680043.8.498.10001344172545964778532427094848882081","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715","series":"1.2.826.0.1.3680043.8.498.10197261968040390389089452666492206520","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066","series":"1.2.826.0.1.3680043.8.498.37713262116361900993748987209685113911","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182","series":"1.2.826.0.1.3680043.8.498.52761229604020904804130025429657006566","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11927352926669104658144933891780551483","series":"1.2.826.0.1.3680043.8.498.87295298734124900567864456125482172502","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11935116791150327546384719051023540185","series":"1.2.826.0.1.3680043.8.498.50713587121784410562449689091388553163","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014","series":"1.2.826.0.1.3680043.8.498.70423130716127470607524449601455088952","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600","series":"1.2.826.0.1.3680043.8.498.28504351096619645180813276294560853490","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11958117172819754862618612170999909480","series":"1.2.826.0.1.3680043.8.498.86363903045677187039144356099110792369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451","series":"1.2.826.0.1.3680043.8.498.11214379029766993573010708654862859536","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11980187124986749407584543247586384443","series":"1.2.826.0.1.3680043.8.498.24925629863365255067700272474449100003","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11981170645605720774509861722988500673","series":"1.2.826.0.1.3680043.8.498.10429487520667803295697537808501483758","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11992172807590757302730513420827032389","series":"1.2.826.0.1.3680043.8.498.84563097252247289502355871436945378999","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473","series":"1.2.826.0.1.3680043.8.498.12192717673477248620712595756402365138","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792","series":"1.2.826.0.1.3680043.8.498.13153837254067338229673605822327557581","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12014959202771141042815134335571931814","series":"1.2.826.0.1.3680043.8.498.65031816209246534930046819953260631494","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122","series":"1.2.826.0.1.3680043.8.498.80187914885730902926858051698333500424","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12036404333388388978317335770611404933","series":"1.2.826.0.1.3680043.8.498.84914170341631094887785090067323770262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12041491173612341571223481616114183729","series":"1.2.826.0.1.3680043.8.498.10841804289140682125136716686873167708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12051169109587120248372991913002670813","series":"1.2.826.0.1.3680043.8.498.63795205963989284926028964198225914441","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12063742398421783626420335095128808816","series":"1.2.826.0.1.3680043.8.498.26917597901638559764793536843459757082","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017","series":"1.2.826.0.1.3680043.8.498.13424389518361772372956394685368606048","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12073401429398042797505820525732905336","series":"1.2.826.0.1.3680043.8.498.11646695881688512265758773171744227365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205","series":"1.2.826.0.1.3680043.8.498.13827169550739691363688317097060606171","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415","series":"1.2.826.0.1.3680043.8.498.10308496537329574421594979493853151721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12110510829608190094905588610131531103","series":"1.2.826.0.1.3680043.8.498.31432270496027343557382340444926867933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761","series":"1.2.826.0.1.3680043.8.498.67959499144143571885439856641643498631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990","series":"1.2.826.0.1.3680043.8.498.12837966021491972334579149948952416610","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12139829070290803654867114374799823711","series":"1.2.826.0.1.3680043.8.498.10532254454926395448894865395739820280","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12153755823296803843819361055386265595","series":"1.2.826.0.1.3680043.8.498.62443695371874371268228550248878411466","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12158630368692962312115489365152319457","series":"1.2.826.0.1.3680043.8.498.19875389692076917978053304777206678998","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045","series":"1.2.826.0.1.3680043.8.498.92861995787799571699733419945236448742","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12168045829469202472124733625621943469","series":"1.2.826.0.1.3680043.8.498.99502610602286431142112913050929041395","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217","series":"1.2.826.0.1.3680043.8.498.11562258343248620698507051102956222844","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12176469158911896593955662590695392044","series":"1.2.826.0.1.3680043.8.498.33877299642220577735539755759654539445","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12181591358572249491885248089245858515","series":"1.2.826.0.1.3680043.8.498.10721890896927490622321254099285924852","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12186494601308291174548814564387764515","series":"1.2.826.0.1.3680043.8.498.79468225810590995450660826380551118735","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12192455583120681639582915365754780698","series":"1.2.826.0.1.3680043.8.498.17482732808116238190223515042208453297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12200672323103829956274910258854933266","series":"1.2.826.0.1.3680043.8.498.10586046133003246843328628324846343353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681","series":"1.2.826.0.1.3680043.8.498.45132915063551105004998917933094082357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986","series":"1.2.826.0.1.3680043.8.498.57229935060360507512473006989665419797","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735","series":"1.2.826.0.1.3680043.8.498.68745831078096610324595520031104270087","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12232127448545053821926845887026738988","series":"1.2.826.0.1.3680043.8.498.73710383637208446795090054113608997141","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12238865510201989224648662550299925120","series":"1.2.826.0.1.3680043.8.498.11910034071413557519847422879658831242","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12242172267695588655664456078003635837","series":"1.2.826.0.1.3680043.8.498.13057481804432714006120300309237838089","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817","series":"1.2.826.0.1.3680043.8.498.14893021698863922043122973077571541107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.43301482233646332005190970516998809676","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.71969054802122081627522096376518395605","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12261175520056712414700866621816882336","series":"1.2.826.0.1.3680043.8.498.11938956612366252573565494112655712659","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12266054872802015665014176014323258615","series":"1.2.826.0.1.3680043.8.498.54977899565663203152167755702987654711","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039","series":"1.2.826.0.1.3680043.8.498.24565939130272277844487353863141995552","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367","series":"1.2.826.0.1.3680043.8.498.98612858546581216405641163370200426456","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12274801503555668613400469130773239683","series":"1.2.826.0.1.3680043.8.498.37091287689764523127897264524393147100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12283662652762881783845275662580546217","series":"1.2.826.0.1.3680043.8.498.18997854213487826853771957790489697831","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12285469859900617016859868410323043734","series":"1.2.826.0.1.3680043.8.498.98630375224746657886712396694707813819","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12294073960417956629256115458135748872","series":"1.2.826.0.1.3680043.8.498.23554049788978726106174763069941081076","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295443776523723913480216844110882937","series":"1.2.826.0.1.3680043.8.498.43197919527922948511906976437541088375","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12301945990161670321760708884248055936","series":"1.2.826.0.1.3680043.8.498.12984412030655226842058048012977672396","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12305281328019552390691110900015627668","series":"1.2.826.0.1.3680043.8.498.13034441341147837590851333927188302472","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12308357850075929758995438211317707857","series":"1.2.826.0.1.3680043.8.498.91319341314520563916238289244473711191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947","series":"1.2.826.0.1.3680043.8.498.19749368529480193769792582433855518204","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914","series":"1.2.826.0.1.3680043.8.498.12263681946668052482489440799400000530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12335573784261147388984372283481919933","series":"1.2.826.0.1.3680043.8.498.80310918362898610890295325771920128724","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12347007036329783363431172840583325937","series":"1.2.826.0.1.3680043.8.498.23300706187624852232483230507770195252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12348960754791091583201828167607825033","series":"1.2.826.0.1.3680043.8.498.66939916851806381827037754969148387256","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12354685390005531691919754048141542648","series":"1.2.826.0.1.3680043.8.498.40740858090125250392660742462763860536","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12356611369549236159671896247094661767","series":"1.2.826.0.1.3680043.8.498.10836818279513067099386617566738342978","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12366238437963147962850341556508931973","series":"1.2.826.0.1.3680043.8.498.13938028542291875098261076514094495644","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790","series":"1.2.826.0.1.3680043.8.498.58245619998479745866904330780307604445","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12381325071746339154385415220972069782","series":"1.2.826.0.1.3680043.8.498.75015429836509057698752703142670885473","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12383716855639375235119728470042467411","series":"1.2.826.0.1.3680043.8.498.60770791939028701874037845243615323685","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","series":"1.2.826.0.1.3680043.8.498.11631126645403399086341632331207064744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12390186846317493876612228407734068726","series":"1.2.826.0.1.3680043.8.498.56757531118731843492950935213399690154","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12406043058701411628555010402350032739","series":"1.2.826.0.1.3680043.8.498.92393028266532455533765289460805789323","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696","series":"1.2.826.0.1.3680043.8.498.64599211287228407360040470705414652990","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831","series":"1.2.826.0.1.3680043.8.498.30612386239966468337355466395385812414","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12426381920136706157912507136730458010","series":"1.2.826.0.1.3680043.8.498.12476206292683557050070170926527943317","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232","series":"1.2.826.0.1.3680043.8.498.13201781095930675225796068553802053648","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876","series":"1.2.826.0.1.3680043.8.498.24507293469595917340226506206208074358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12450858400030959723767712312663176092","series":"1.2.826.0.1.3680043.8.498.53087114985556059304179860860983303940","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080","series":"1.2.826.0.1.3680043.8.498.69613354161903871112117584221660122246","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12473755232610784318190599222252893949","series":"1.2.826.0.1.3680043.8.498.31747750103246053320462646501126449916","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084","series":"1.2.826.0.1.3680043.8.498.12998890517338350488146486681583721992","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.17732950737538175330566324448434448072","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348","series":"1.2.826.0.1.3680043.8.498.12522271809876130017509595648823268615","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.59233436666055342259713041291708777766","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072","series":"1.2.826.0.1.3680043.8.498.43349730538110110071531484201970400089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919","series":"1.2.826.0.1.3680043.8.498.12320101971011590040393044930016612133","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12509825237087140646629745447753158673","series":"1.2.826.0.1.3680043.8.498.90193208326896331242722403033174927037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12514357232100339485873453417784586592","series":"1.2.826.0.1.3680043.8.498.92185072554366374681573734389665647931","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12520203592477518476890010185037272575","series":"1.2.826.0.1.3680043.8.498.12643426552459715411686892147808832710","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12523676732876902778577970655429060858","series":"1.2.826.0.1.3680043.8.498.99832742199528318779635947080225309837","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12526590714362529606046044104426807158","series":"1.2.826.0.1.3680043.8.498.38353516881417388658951800382745896452","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","series":"1.2.826.0.1.3680043.8.498.12458127854903271191938201774920464921","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111","series":"1.2.826.0.1.3680043.8.498.69936680178883113424329839971087813818","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12552701911592685137269544109687472988","series":"1.2.826.0.1.3680043.8.498.88603941294997627371423929656153015565","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12554824497579383319980187351290254727","series":"1.2.826.0.1.3680043.8.498.66418508538237909836735017393708148660","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395","series":"1.2.826.0.1.3680043.8.498.98716913977099930465872971249195280692","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.97349071835893908339382328969818985158","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12574270269124650675934073761741661292","series":"1.2.826.0.1.3680043.8.498.88926092114271925298258539086062885982","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431","series":"1.2.826.0.1.3680043.8.498.95888677565579339893553769811147859768","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12583872770399266463538638801797827468","series":"1.2.826.0.1.3680043.8.498.87674189798986780550750885797184545833","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.27146062832035348929206890381154881403","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607659092357834912029375690348886159","series":"1.2.826.0.1.3680043.8.498.12803037699082480306633482790012748409","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574","series":"1.2.826.0.1.3680043.8.498.10618719904685447879407202593946857928","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12620718517255799151283445360332799702","series":"1.2.826.0.1.3680043.8.498.64116819661997497410286342283209917503","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12629372332421954368597330267888483345","series":"1.2.826.0.1.3680043.8.498.11527865501736554141973679796410978498","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.28256183365247136519637095211742683797","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719","series":"1.2.826.0.1.3680043.8.498.97896267720727218829963969902237582095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12642641661988102031826128876502303807","series":"1.2.826.0.1.3680043.8.498.31436448016906360396684483720000966768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265","series":"1.2.826.0.1.3680043.8.498.10838418400700539871832977591918853318","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625","series":"1.2.826.0.1.3680043.8.498.59261179613172600683863119655142813082","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12668708646283106269005674386248135220","series":"1.2.826.0.1.3680043.8.498.18280374154128929454282810621768235404","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12674930973831274527829261576087154644","series":"1.2.826.0.1.3680043.8.498.43967184186559311591255687928402974794","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12688954493547892047021455671738310186","series":"1.2.826.0.1.3680043.8.498.35593770069961139464945530424863643598","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.43475452295441929303058000527885521475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111","series":"1.2.826.0.1.3680043.8.498.10097242914443185772462346268255956363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803","series":"1.2.826.0.1.3680043.8.498.83660473892709974828983408413080365147","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12709854414521475738435319009767427434","series":"1.2.826.0.1.3680043.8.498.12819671759571399478050654645611362386","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12715402081094788880283826571710131940","series":"1.2.826.0.1.3680043.8.498.36119895598969135147149978562563124427","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12718966639501121738042125605376714231","series":"1.2.826.0.1.3680043.8.498.33261842750988901379684604020035910159","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405","series":"1.2.826.0.1.3680043.8.498.89154160764207938031123586597974798772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12727488524631885827798129418558343386","series":"1.2.826.0.1.3680043.8.498.62370362993845453576068065706890010581","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492","series":"1.2.826.0.1.3680043.8.498.69865184058804971315815142769051482779","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12759704450656741777120047828560448622","series":"1.2.826.0.1.3680043.8.498.68202932762909658171148120227708849687","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361","series":"1.2.826.0.1.3680043.8.498.10472185304269409759432576707469508079","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908","series":"1.2.826.0.1.3680043.8.498.19261027124134078986327642344057207120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270","series":"1.2.826.0.1.3680043.8.498.26766220543859727250235595384754948016","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333","series":"1.2.826.0.1.3680043.8.498.12117257786583224586651480136829468269","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802","series":"1.2.826.0.1.3680043.8.498.13307048612425104255296154491239759601","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532","series":"1.2.826.0.1.3680043.8.498.12151380225317193011096785072795607515","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12799849540231315910448269889175806934","series":"1.2.826.0.1.3680043.8.498.41490353144437957688879751827923188894","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492","series":"1.2.826.0.1.3680043.8.498.51148402259712862712353546920527079297","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043","series":"1.2.826.0.1.3680043.8.498.57710064925203569769013241485921210200","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996","series":"1.2.826.0.1.3680043.8.498.72591720011372335977352290412127385233","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174","series":"1.2.826.0.1.3680043.8.498.89528498323824870132465222598690750362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751","series":"1.2.826.0.1.3680043.8.498.19075761505144797953980529582078068138","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052","series":"1.2.826.0.1.3680043.8.498.26766892128996980912616275088898755837","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12839498544183083639548298117995964432","series":"1.2.826.0.1.3680043.8.498.11949371831201026504649403315938747879","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12864336642218457767597053831458716404","series":"1.2.826.0.1.3680043.8.498.12106915681211981688818788391945490318","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502","series":"1.2.826.0.1.3680043.8.498.61348928870156735601280147624546194437","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832","series":"1.2.826.0.1.3680043.8.498.55488258039309888461878387883525775045","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.89728513777475526676213642603012503104","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12919303621342250989179219420795277559","series":"1.2.826.0.1.3680043.8.498.88179162571862140577226411882042090799","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12936296973456045538701035923586653502","series":"1.2.826.0.1.3680043.8.498.99530910487841282080925925641871850306","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12942677240800343332600884989364651588","series":"1.2.826.0.1.3680043.8.498.71750803807363960647988355499260752822","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12949322114105642048256996295428966924","series":"1.2.826.0.1.3680043.8.498.79060555373017460085867265389261340872","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935","series":"1.2.826.0.1.3680043.8.498.44233781742222524987884796773447110882","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12958999546496283565938189274771483491","series":"1.2.826.0.1.3680043.8.498.86674778335726389716928609392480973030","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626","series":"1.2.826.0.1.3680043.8.498.28967198394453768027381264979888648956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12978172997053500029791786614906570285","series":"1.2.826.0.1.3680043.8.498.64731864976042324232958018933306960897","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.11702937130191473140486762779513429126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12985683732818911488433522743768046254","series":"1.2.826.0.1.3680043.8.498.11300602227430919389514792325314379023","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12986517636774565418070546261705439006","series":"1.2.826.0.1.3680043.8.498.32937638522682299723926407074801342881","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12992013816638129896738209015383985569","series":"1.2.826.0.1.3680043.8.498.35076325044576605294591097388552356125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13009271274636013067662786774946029285","series":"1.2.826.0.1.3680043.8.498.93231024109273104527730286606347716797","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13016684730504329799460698013153308088","series":"1.2.826.0.1.3680043.8.498.55116087119309805586712312179575108914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13022585036130454941964190427558363602","series":"1.2.826.0.1.3680043.8.498.10223745295250224015482174590927847450","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.20846461297451162567782491058460444160","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852","series":"1.2.826.0.1.3680043.8.498.11715899933342663562229392903419213421","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13028226019192925363616760332442456953","series":"1.2.826.0.1.3680043.8.498.21159904756356784281128612320515918410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13035632998746261521316395720102873141","series":"1.2.826.0.1.3680043.8.498.31199299811468265897229786865570943232","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13054766954092919774634499780241210835","series":"1.2.826.0.1.3680043.8.498.11809880785037741727076940023068911522","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13065120731888368919957856813755701101","series":"1.2.826.0.1.3680043.8.498.51812844484454561109021098615263143810","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13072908567275370866813748886351904593","series":"1.2.826.0.1.3680043.8.498.52382707741487705470038819275802986713","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13077357855769903335728068420884303993","series":"1.2.826.0.1.3680043.8.498.17773658642155245126575608759427592095","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421","series":"1.2.826.0.1.3680043.8.498.15120211142691490649956182284473931249","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749","series":"1.2.826.0.1.3680043.8.498.11321740822315247610326816826118591840","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13088550794087945662322609097092458487","series":"1.2.826.0.1.3680043.8.498.11074249433466093812155625661119382994","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456","series":"1.2.826.0.1.3680043.8.498.98098538398813067689620821965125962097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13099702881219862576749493772702888064","series":"1.2.826.0.1.3680043.8.498.31320092523329916417093115484432707749","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590","series":"1.2.826.0.1.3680043.8.498.47458708658125695577865990859859575659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13114228294359876639109410575915394712","series":"1.2.826.0.1.3680043.8.498.58119635651896158459137109726762464267","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285","series":"1.2.826.0.1.3680043.8.498.50158291457838672411049691466821459248","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13125910783461518266627225743585018143","series":"1.2.826.0.1.3680043.8.498.91796868207380976499476756080823748183","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048","series":"1.2.826.0.1.3680043.8.498.56962213025560925595800474549011461941","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773","series":"1.2.826.0.1.3680043.8.498.49154629138528309010124784669076423324","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258","series":"1.2.826.0.1.3680043.8.498.65158661839278012635546349258879632500","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.48081422183613566932764217548690553164","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545","series":"1.2.826.0.1.3680043.8.498.11750428867198073970737667410261445943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13139684164906828131777150061532685584","series":"1.2.826.0.1.3680043.8.498.12091848908136620650381662198236058331","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13146466304138609448483938844487385775","series":"1.2.826.0.1.3680043.8.498.56192160105769036462806696590556890208","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13150780733560495803861509587935093776","series":"1.2.826.0.1.3680043.8.498.25467967891917557016081913876022855916","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956","series":"1.2.826.0.1.3680043.8.498.82021453544799579721293714933489234699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13157776420553094741176082047621957775","series":"1.2.826.0.1.3680043.8.498.10811413155349065586855105664860054185","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296","series":"1.2.826.0.1.3680043.8.498.76217807699648382049716887478040828227","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13162479706670045075896993021883562680","series":"1.2.826.0.1.3680043.8.498.86007236177648109485962183258951081408","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13167727300538353052482338786183678684","series":"1.2.826.0.1.3680043.8.498.10994427898459458953271906694058874374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13173050135651736408720687361834045815","series":"1.2.826.0.1.3680043.8.498.26888972423610830962426547991122357124","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937","series":"1.2.826.0.1.3680043.8.498.10534755788761625628788751288900830497","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13179829911816163852815103052496326166","series":"1.2.826.0.1.3680043.8.498.88106538093004054076589217504512105002","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13200225436827306542001129042739578441","series":"1.2.826.0.1.3680043.8.498.10790054971327625184519273126809670949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145","series":"1.2.826.0.1.3680043.8.498.12797237278706555140493717270551268212","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605","series":"1.2.826.0.1.3680043.8.498.21010088108642542466968346808288244341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13222515831888962408632292783773537312","series":"1.2.826.0.1.3680043.8.498.12828070682257087394513659544802098446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477","series":"1.2.826.0.1.3680043.8.498.12375589615941521155462583599140879064","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.57191162647593192902676660927764700297","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308","series":"1.2.826.0.1.3680043.8.498.11362645941562353234152333408026635306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13256173302993031630968402868348974108","series":"1.2.826.0.1.3680043.8.498.71821696405509259217113512165047183360","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262473718294425031790270527413368245","series":"1.2.826.0.1.3680043.8.498.88331617024988433105064362711020807627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733","series":"1.2.826.0.1.3680043.8.498.75819317201549022775955801449706477737","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636","series":"1.2.826.0.1.3680043.8.498.37529407673758290376377411313488967862","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13280017707177438711374726939683941320","series":"1.2.826.0.1.3680043.8.498.92195222450971502838522384180211190505","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662","series":"1.2.826.0.1.3680043.8.498.26260895244099527901547331681734574474","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.11322695511401983536538565837577264736","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.26462954853422926669750092454094458604","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13309299049346932814529962458932580753","series":"1.2.826.0.1.3680043.8.498.10170666500069174010174315542068695060","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13314276151860609710431717811242659651","series":"1.2.826.0.1.3680043.8.498.27284292534375879013741568351587773492","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13323187196249612936800308722675884644","series":"1.2.826.0.1.3680043.8.498.35373079750417088735206850985213830685","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121","series":"1.2.826.0.1.3680043.8.498.12564642579734401006580124874119395630","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13335786652653204791740906118815827207","series":"1.2.826.0.1.3680043.8.498.60583547196913159995296445885606476963","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13353249891620291438488727876474802879","series":"1.2.826.0.1.3680043.8.498.57788344109796400009097996455510220328","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13357923062946813820390731946535198170","series":"1.2.826.0.1.3680043.8.498.18334568037433683791485863369299078962","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567","series":"1.2.826.0.1.3680043.8.498.45914850447024909975712101263344365271","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13367691581890020433564928882602648123","series":"1.2.826.0.1.3680043.8.498.50713419529352560439676464101130689237","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13369443704765101449274062863864122737","series":"1.2.826.0.1.3680043.8.498.56158331754353194550979269342894895166","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.80917273700700432825726526156869574714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13383972749798909973472489456126521194","series":"1.2.826.0.1.3680043.8.498.10172096675742181949603061448097219152","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680","series":"1.2.826.0.1.3680043.8.498.13298066112417166769462783253486447206","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13400273795954308944359576841942016981","series":"1.2.826.0.1.3680043.8.498.73444258677281233003893935195578871849","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965","series":"1.2.826.0.1.3680043.8.498.10202970152092055869296214177741879055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13459473184684263663596121399737033933","series":"1.2.826.0.1.3680043.8.498.34253385685654200198680084421361366124","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.66386686775701547745538907266652789270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13505671694940677161128783863912245059","series":"1.2.826.0.1.3680043.8.498.48394330272708442161187884475172778393","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636","series":"1.2.826.0.1.3680043.8.498.12591062095248720552765137708795655428","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179","series":"1.2.826.0.1.3680043.8.498.70974275854578491464130711650519926524","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13762716088334302039143651507177887969","series":"1.2.826.0.1.3680043.8.498.97986291329456633657104469386786421259","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13879163301346984697246311226647565712","series":"1.2.826.0.1.3680043.8.498.11581687063757203840414746282562203591","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13958765761660666865188384031150827763","series":"1.2.826.0.1.3680043.8.498.13003221383272112675989892449572901629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14024129019390041106306062001538710978","series":"1.2.826.0.1.3680043.8.498.10652747667138737825221820590139327506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847","series":"1.2.826.0.1.3680043.8.498.45583994966048702246857239948856593540","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14137431039800236510749401303688858447","series":"1.2.826.0.1.3680043.8.498.66017946494797772893237206258217008757","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14218113768651197803980865495163189825","series":"1.2.826.0.1.3680043.8.498.12897606105317706334371603741726663338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14300107110014472682373589824643789093","series":"1.2.826.0.1.3680043.8.498.90940965337890969138779175265352480876","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575","series":"1.2.826.0.1.3680043.8.498.10627679953367517598327319559383704808","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532","series":"1.2.826.0.1.3680043.8.498.11374002494404109454300818188155187830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166","series":"1.2.826.0.1.3680043.8.498.52495983443683473342274272028843667017","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14490539215064287912254286565666891557","series":"1.2.826.0.1.3680043.8.498.50065443576755616988034731165180619296","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14558486491688263090686005096911018438","series":"1.2.826.0.1.3680043.8.498.23947477787937739977289496094626179953","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14728068688166107022558957899468489735","series":"1.2.826.0.1.3680043.8.498.76310028721186419699875866622664293236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14776241668061173271058268311868302014","series":"1.2.826.0.1.3680043.8.498.42714212391331389541847421397344496315","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770","series":"1.2.826.0.1.3680043.8.498.99132457810631721249766263252874046183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14959561028578335448289993045623217861","series":"1.2.826.0.1.3680043.8.498.12584829957386474662805353837712778160","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904","series":"1.2.826.0.1.3680043.8.498.10002649128333294739274777073688197429","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15065556607080211420503043544482923211","series":"1.2.826.0.1.3680043.8.498.80043237350410960959365086204262222794","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15103255966832759925770334719703425755","series":"1.2.826.0.1.3680043.8.498.93615777377908278429736137037566217022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247","series":"1.2.826.0.1.3680043.8.498.68922083330902432577368780411177185647","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15326951535563258414956521742264374802","series":"1.2.826.0.1.3680043.8.498.11067243536734581253215476721829286862","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15376535778246978844757121776560020722","series":"1.2.826.0.1.3680043.8.498.13392970546321716351091444982694439362","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351","series":"1.2.826.0.1.3680043.8.498.11351272094114649644789984461230635610","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269","series":"1.2.826.0.1.3680043.8.498.14302568917510290107849177794535792261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15539098084847502447238233198010208924","series":"1.2.826.0.1.3680043.8.498.68107273020870930268497329233186999377","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.15595595490910196952568459534537629865","series":"1.2.826.0.1.3680043.8.498.50983021259128645653238651398524487124","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15709232000390534671340322807442769184","series":"1.2.826.0.1.3680043.8.498.31548687913564761340581463554606841839","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15851463019383290210760064201595021029","series":"1.2.826.0.1.3680043.8.498.19046773885152622092177871045455595661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16046164723877343553376490423345098256","series":"1.2.826.0.1.3680043.8.498.11097191250036571480459781109658421061","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.10114318007378072154267323023488130369","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081","series":"1.2.826.0.1.3680043.8.498.34375250786030727026773690676301401790","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16200518336931321995379999168737787921","series":"1.2.826.0.1.3680043.8.498.15584982064278417309946996204089363433","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452","series":"1.2.826.0.1.3680043.8.498.45784664706509355482170729251436127904","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16364873390453802195568603605191835957","series":"1.2.826.0.1.3680043.8.498.24054545577548250366170951634210271915","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16469256144078887607437578587242258846","series":"1.2.826.0.1.3680043.8.498.26302070602049575401109396480371143715","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16527747624688737126118546487228060848","series":"1.2.826.0.1.3680043.8.498.22203546380636106185439174376794113928","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","series":"1.2.826.0.1.3680043.8.498.11935452045498234015211521099504596541","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350","series":"1.2.826.0.1.3680043.8.498.12020283710610650290858943789978589013","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16674237773422474998796191258459643061","series":"1.2.826.0.1.3680043.8.498.13389878716352694674323700462905547322","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779","series":"1.2.826.0.1.3680043.8.498.60870146417630837080908442138538884268","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16724013341359693140331158148551346369","series":"1.2.826.0.1.3680043.8.498.63952818381988713704370543580975175984","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16843381581429596234606436682284624187","series":"1.2.826.0.1.3680043.8.498.89186693694269795828987544534570473115","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16862596867845295946732698348903803700","series":"1.2.826.0.1.3680043.8.498.79004110790832233680641230988168636551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17049315826418217889213226790962592213","series":"1.2.826.0.1.3680043.8.498.55362744780985950088090964573087688681","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835","series":"1.2.826.0.1.3680043.8.498.14465993184755334109002726611443329352","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.88593448085647941227670430637249380422","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17123244662958741977525806700986256210","series":"1.2.826.0.1.3680043.8.498.72988416210692759186650256264378268453","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17160610480070701553118850947260201608","series":"1.2.826.0.1.3680043.8.498.51892990661212985818484235887327769830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509","series":"1.2.826.0.1.3680043.8.498.86865248899456675729455906824324163570","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823","series":"1.2.826.0.1.3680043.8.498.11350330383955415028526030673123341678","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519","series":"1.2.826.0.1.3680043.8.498.72999316872567856854418822737486337441","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17547781127981007962008163248942714330","series":"1.2.826.0.1.3680043.8.498.87311733822532473930767430137150449037","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17662880041629379035049367252086125592","series":"1.2.826.0.1.3680043.8.498.62416699430061763998803994727864811090","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17691941055331862184525411493110144433","series":"1.2.826.0.1.3680043.8.498.96858718945465959561625853152537364727","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.12023525097305968650090822258728072936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17836815413824722543229791044312174885","series":"1.2.826.0.1.3680043.8.498.65390092992629763088019505823185515105","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366","series":"1.2.826.0.1.3680043.8.498.94935299353902630816852012637759140008","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17931021655186901394796168474942485728","series":"1.2.826.0.1.3680043.8.498.10003074389531472338522226501097852466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730","series":"1.2.826.0.1.3680043.8.498.72043274517048913626950162982553609232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18092766733837308394607532608319746043","series":"1.2.826.0.1.3680043.8.498.90772753229665327790393430816216114035","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285","series":"1.2.826.0.1.3680043.8.498.92270962245482103345527330259843730105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584","series":"1.2.826.0.1.3680043.8.498.10298812622325318054519069066426733792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080","series":"1.2.826.0.1.3680043.8.498.51466075619932069705286626555795289537","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067","series":"1.2.826.0.1.3680043.8.498.98889380449897024049951782454821556994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908","series":"1.2.826.0.1.3680043.8.498.90964443019788741204119188021117429911","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18518393141873346710197645305647124682","series":"1.2.826.0.1.3680043.8.498.10869310324248929747525445715785270678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18614704377104425868545800677445972261","series":"1.2.826.0.1.3680043.8.498.77875466757290385117183404483614403077","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748","series":"1.2.826.0.1.3680043.8.498.99426969875949206866046059439521487089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006","series":"1.2.826.0.1.3680043.8.498.54398966811683622808940154491468694802","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566","series":"1.2.826.0.1.3680043.8.498.10121571082663841590511644532713888380","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.18001431528267381721610357343188054334","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18848481788456572234982139368351538716","series":"1.2.826.0.1.3680043.8.498.53615371198251410196893186804193697476","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18931107448953327254073392032141677797","series":"1.2.826.0.1.3680043.8.498.53530571948661770291567391338132735107","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874","series":"1.2.826.0.1.3680043.8.498.14434123796539886253952186691414701140","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19124879088354547836118593585646705198","series":"1.2.826.0.1.3680043.8.498.53761027211682777482376557459709147455","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19183011851352129901149443549646899548","series":"1.2.826.0.1.3680043.8.498.37063655450541533957480528191284896024","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19198152070740205434848422424805073748","series":"1.2.826.0.1.3680043.8.498.12078297686559548116416952474717484278","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19342628541833173544083802327412975225","series":"1.2.826.0.1.3680043.8.498.21279178053662075723771322799056978986","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19439950168933299231685815631786334293","series":"1.2.826.0.1.3680043.8.498.27873965485360113893510533293558569244","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19489875090732464985703624680756212359","series":"1.2.826.0.1.3680043.8.498.77703733877193279265246773795281258502","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19573195389434178432956976770560221462","series":"1.2.826.0.1.3680043.8.498.69482908092919995365799694884562884380","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296","series":"1.2.826.0.1.3680043.8.498.69602439266530826431206442120752978810","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031","series":"1.2.826.0.1.3680043.8.498.60975969724759000684070553639677081557","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955","series":"1.2.826.0.1.3680043.8.498.54494516519889180804460372904088862999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19832308119617081676615236082965891208","series":"1.2.826.0.1.3680043.8.498.51017870623731928166957772603683046990","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760","series":"1.2.826.0.1.3680043.8.498.61358459936414246387457682094924330891","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19925337578871911327225897926650188274","series":"1.2.826.0.1.3680043.8.498.49245794465960565217368275009886989457","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19977273190231806393528180968361858122","series":"1.2.826.0.1.3680043.8.498.90379137953625930379424850396961647525","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20123813962665544845986028832159432587","series":"1.2.826.0.1.3680043.8.498.77913005389977338744237972910868903984","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20184541870535223885310764061146422870","series":"1.2.826.0.1.3680043.8.498.75945280503014786759859884763802538882","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20196165276643741937915346646793735100","series":"1.2.826.0.1.3680043.8.498.33058179361288258855688941684882199643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20266523169651039493799975212435349766","series":"1.2.826.0.1.3680043.8.498.78617525683344834381183520615502165328","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20301796413163444642416918108994411350","series":"1.2.826.0.1.3680043.8.498.22734719783990648495819260669365401671","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20427402498958253193571475926580142499","series":"1.2.826.0.1.3680043.8.498.12716292788378494009872026950603787897","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.52504327050979558401346566753733502381","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.11452430246317841725938599513849052358","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20587514285427671071934631922521946816","series":"1.2.826.0.1.3680043.8.498.11523217572126152962160921544165440166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20690227395174139480327772367312879333","series":"1.2.826.0.1.3680043.8.498.37980773824570750788020307139277092853","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784","series":"1.2.826.0.1.3680043.8.498.10749787484808383131258619247430599358","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20852351697358589606507109300359572624","series":"1.2.826.0.1.3680043.8.498.11996826148648162635921652499436529095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270","series":"1.2.826.0.1.3680043.8.498.56844460485952340556089632287467034173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615","series":"1.2.826.0.1.3680043.8.498.12139928432601333888203434291422290560","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21024377993947321291280779867493407199","series":"1.2.826.0.1.3680043.8.498.20929941992743037009773327429678095606","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21135201439483157418374033420156525658","series":"1.2.826.0.1.3680043.8.498.10088722197008341424800966901103293591","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849","series":"1.2.826.0.1.3680043.8.498.12421986627737156314377920533884692694","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596","series":"1.2.826.0.1.3680043.8.498.25979081878412993668470183332414919751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21403015320378798778538823075316902483","series":"1.2.826.0.1.3680043.8.498.95809182289631106242388773019224657783","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.19944191560206457855272502954921085214","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.21556073772857867149576001337847113661","series":"1.2.826.0.1.3680043.8.498.10308612330982861991719143549300155186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044","series":"1.2.826.0.1.3680043.8.498.12337631723996443473990223021624773188","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21634792491972658759276025394685307053","series":"1.2.826.0.1.3680043.8.498.86471309283202983932030505791954142461","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21769352980698937182060933429354012104","series":"1.2.826.0.1.3680043.8.498.10190134994957865797837272671196513303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21868734450232158209414868901539475548","series":"1.2.826.0.1.3680043.8.498.96109445896342609621668232787961895460","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21904421326144238681996713485932410596","series":"1.2.826.0.1.3680043.8.498.10926920291479903576333878541630954858","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762","series":"1.2.826.0.1.3680043.8.498.13342761481170638701155471613070779961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537","series":"1.2.826.0.1.3680043.8.498.10463711754458358542500176827160593489","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22053727963380087706894183799325067544","series":"1.2.826.0.1.3680043.8.498.12435949525592914447965909941729928072","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404","series":"1.2.826.0.1.3680043.8.498.71247426330737891374731581744560399312","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22126824950877890715014783370071292997","series":"1.2.826.0.1.3680043.8.498.10122763195023616268374570020881825465","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844","series":"1.2.826.0.1.3680043.8.498.10027316035548781677096498300768910747","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643","series":"1.2.826.0.1.3680043.8.498.11568485591159278024419822096638847130","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22394208178597637441149549895512290013","series":"1.2.826.0.1.3680043.8.498.60096758608120610360050119878807545373","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645","series":"1.2.826.0.1.3680043.8.498.13138486775571230585276087000203802683","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060","series":"1.2.826.0.1.3680043.8.498.84877775535105040027936859387203957589","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22583985355672251619550173090948161714","series":"1.2.826.0.1.3680043.8.498.11877281930598952320725247364074742484","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22594742481654038247097269358877852869","series":"1.2.826.0.1.3680043.8.498.11300945803954158048809453598651474121","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020","series":"1.2.826.0.1.3680043.8.498.99368107790978989121257524002195260591","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22678957404909201599340396039237525550","series":"1.2.826.0.1.3680043.8.498.10016262976055443755653232919583713642","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22867776500261605752195206933939172136","series":"1.2.826.0.1.3680043.8.498.39399060706827096468906813787506824456","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22938598188283919466040132624303635817","series":"1.2.826.0.1.3680043.8.498.82455263833927974735763308466906170953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449","series":"1.2.826.0.1.3680043.8.498.71176307654690899719678972540948510644","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23063675880057833741455102729816364091","series":"1.2.826.0.1.3680043.8.498.26395964187297215523020077545855554069","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416","series":"1.2.826.0.1.3680043.8.498.71143302854795877471055145797891292249","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23107903961514228607587963030481481744","series":"1.2.826.0.1.3680043.8.498.10875220221313346496926201774349889539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738","series":"1.2.826.0.1.3680043.8.498.12814814131876601312700547145155475094","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23422918444346962998731376196027420200","series":"1.2.826.0.1.3680043.8.498.13656190730632248805206204359272353482","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23518857399095666263158269685495375084","series":"1.2.826.0.1.3680043.8.498.64920530860400150884785215149984342392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899","series":"1.2.826.0.1.3680043.8.498.32777980175611390481454327384408215287","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23719982492392272262684855675016475265","series":"1.2.826.0.1.3680043.8.498.10817169823328511005253820519156142622","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23757007145576852496599816072431075057","series":"1.2.826.0.1.3680043.8.498.65085044117120766461723701836628859834","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23849672490426589132734390202638283865","series":"1.2.826.0.1.3680043.8.498.10307576786978928446860717171819607793","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277","series":"1.2.826.0.1.3680043.8.498.64377746410115163216126275635225648263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24012076387083114283116476459425015413","series":"1.2.826.0.1.3680043.8.498.32928736187430126947384119461046304243","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916","series":"1.2.826.0.1.3680043.8.498.11046858953591803686991180687662050663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24219103446788192052503003052696523200","series":"1.2.826.0.1.3680043.8.498.46997994626264052333012734908948888662","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405","series":"1.2.826.0.1.3680043.8.498.12634783225883900224682749655019619425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24302889622350917085868032747474929504","series":"1.2.826.0.1.3680043.8.498.42440360985313207548754406271016306559","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24432243084670627958434695475653885175","series":"1.2.826.0.1.3680043.8.498.76871694582080776526854506659109687338","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24502332819381293694020055836255028614","series":"1.2.826.0.1.3680043.8.498.14958379466287386130215219527282431060","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24563801799946310644914912532825629118","series":"1.2.826.0.1.3680043.8.498.10941408198935855267565325596547171335","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24635589616467409099788838058892886889","series":"1.2.826.0.1.3680043.8.498.79483590643870138748376315678284207403","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24680736086205072406257935876858402593","series":"1.2.826.0.1.3680043.8.498.55350139277152985054131652622537552199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24808246532191535554375304081795021306","series":"1.2.826.0.1.3680043.8.498.10756628319361859592316861411478317055","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720","series":"1.2.826.0.1.3680043.8.498.27609550851675570457788472401182920826","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.93545638459202367257906011312059571619","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25010661650105533302140829168731585935","series":"1.2.826.0.1.3680043.8.498.26422782538212168973224039348895059691","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861","series":"1.2.826.0.1.3680043.8.498.10675114465276172491992868074541088569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25156207169523051276979595262558652072","series":"1.2.826.0.1.3680043.8.498.80152301548443414553843754351301686976","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25196393144875332073285250936506555534","series":"1.2.826.0.1.3680043.8.498.89316309896784052949993574455796927731","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724","series":"1.2.826.0.1.3680043.8.498.40514037118971314915376302267096888431","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319","series":"1.2.826.0.1.3680043.8.498.11883880238613598804673035629634892046","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25394815405242666778213609728680620077","series":"1.2.826.0.1.3680043.8.498.80028633841746797775060258799675518691","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.54932324706571970780541989427552408874","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25505153402978560400776672230220616863","series":"1.2.826.0.1.3680043.8.498.11419708342178074344805985802498423067","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25550871519416839807700061317524004550","series":"1.2.826.0.1.3680043.8.498.88474165052433157389022680567291267437","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25604547074218147115077644853162295379","series":"1.2.826.0.1.3680043.8.498.23076657557607529973309424638785804456","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25643897330283020467388814112594686690","series":"1.2.826.0.1.3680043.8.498.28003827974302032585895344554915082666","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567","series":"1.2.826.0.1.3680043.8.498.35611592781124849086770360793700534578","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.37209366517316832302582735089595101573","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103","series":"1.2.826.0.1.3680043.8.498.49693491040165732043185305887827795518","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25766387275525303860953321689718246405","series":"1.2.826.0.1.3680043.8.498.70930463164067681093740814539475625731","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25815857653380672532983223088817283976","series":"1.2.826.0.1.3680043.8.498.11252821884951370073263563558394784591","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502","series":"1.2.826.0.1.3680043.8.498.80730417640362255223089505985658006276","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25909078145370189195643713208718315003","series":"1.2.826.0.1.3680043.8.498.22791354835813343173753788611622051828","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","series":"1.2.826.0.1.3680043.8.498.11151785959493076776526064670326531325","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26004572902429209855865822977387283547","series":"1.2.826.0.1.3680043.8.498.12012915444740647304244179341773572841","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26066524302907849854339129271283426208","series":"1.2.826.0.1.3680043.8.498.12042643404332825962060061473520124781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568","series":"1.2.826.0.1.3680043.8.498.50492466650202735226750549412655629090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091","series":"1.2.826.0.1.3680043.8.498.85037406443627175359561191220597507119","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26333849754345698117396119042432557445","series":"1.2.826.0.1.3680043.8.498.85537931925911391167271099481172811399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26408081008482310696738068933718440200","series":"1.2.826.0.1.3680043.8.498.81438689137845392819505542471113150309","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691","series":"1.2.826.0.1.3680043.8.498.25311091424978583393486009541823685719","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.72196561475497299555911757004915131092","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957","series":"1.2.826.0.1.3680043.8.498.27438734623712189369981075224869703141","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26725172078893286313351418435377708028","series":"1.2.826.0.1.3680043.8.498.97180010542738205712438442926064461192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26842258746390750314059176647398369242","series":"1.2.826.0.1.3680043.8.498.71862395983913257546145082389247848927","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26848412380762311150802301982233072694","series":"1.2.826.0.1.3680043.8.498.70521424925419104507233916431321910314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26946980494141305258556994153410856428","series":"1.2.826.0.1.3680043.8.498.40766117580075519310751931175900143409","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26989584479346775218799121753035575730","series":"1.2.826.0.1.3680043.8.498.94275894786284219518779660898421068326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27025989808002468335294585261291982661","series":"1.2.826.0.1.3680043.8.498.64065389598762387303480867295493288502","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27157074709305277871045901905120405881","series":"1.2.826.0.1.3680043.8.498.11402425284792213921206152025667337655","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858","series":"1.2.826.0.1.3680043.8.498.96512257644808009784090669576921272298","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27191574009501966309382307682020135126","series":"1.2.826.0.1.3680043.8.498.18712030622582066168405613486084424934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27197413293500684424735285039923168926","series":"1.2.826.0.1.3680043.8.498.13017390175403382006093918637762022479","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545","series":"1.2.826.0.1.3680043.8.498.11005397036885713173471007818157207010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.81756564566876084705240449374884767561","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486","series":"1.2.826.0.1.3680043.8.498.81981702592814127817584517116104250261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924","series":"1.2.826.0.1.3680043.8.498.17611332847226601740186107923664950771","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27460982183399364904497832933739191826","series":"1.2.826.0.1.3680043.8.498.42741429838102553076111035578512946234","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","series":"1.2.826.0.1.3680043.8.498.29108730891665036523151842031055423943","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27653320597564308144389724321115208375","series":"1.2.826.0.1.3680043.8.498.11229247696285308722909597631050020097","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780","series":"1.2.826.0.1.3680043.8.498.23711895519764225069822163214978457199","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27710718386318170070605046744941324421","series":"1.2.826.0.1.3680043.8.498.45572999019295825925037024858934860609","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723573213501259053792522608339812759","series":"1.2.826.0.1.3680043.8.498.45996460900675566833881045861517869435","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27793455627426963285485957211105588905","series":"1.2.826.0.1.3680043.8.498.92169048048224785932525369156937674695","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935","series":"1.2.826.0.1.3680043.8.498.93812965768041296260506404742557386138","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449","series":"1.2.826.0.1.3680043.8.498.63889474798356709732628924099666752129","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27990965288663885936880638302158503894","series":"1.2.826.0.1.3680043.8.498.48470005290588239824099577189666047428","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28023795363827851445172311810918021563","series":"1.2.826.0.1.3680043.8.498.72091973818363546479981168216630065406","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325","series":"1.2.826.0.1.3680043.8.498.47426593762316137625614430834307564778","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241","series":"1.2.826.0.1.3680043.8.498.12514937975779209595478943363175137438","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28220626968612492259133318770141901567","series":"1.2.826.0.1.3680043.8.498.11060488300907106055453001110990438939","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882","series":"1.2.826.0.1.3680043.8.498.10109044440799198643532539309158324287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28295265939946677460838974253299313798","series":"1.2.826.0.1.3680043.8.498.67564495309077141438923881192893482688","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525","series":"1.2.826.0.1.3680043.8.498.36007956206567840613457153587604291748","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098","series":"1.2.826.0.1.3680043.8.498.10996320913706997581656197411371765557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737","series":"1.2.826.0.1.3680043.8.498.45984057314421837330184846819766405862","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28579372595938810820078670002733005121","series":"1.2.826.0.1.3680043.8.498.53956107821699482663636498615832467888","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156","series":"1.2.826.0.1.3680043.8.498.32661878426112312349467005827092773257","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28638490997780173004029312240334483157","series":"1.2.826.0.1.3680043.8.498.96809673382535983344686466371345362763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28655385212214986812679045835025619123","series":"1.2.826.0.1.3680043.8.498.39823041224413277480268573209405009630","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456","series":"1.2.826.0.1.3680043.8.498.10619932062477675260987333564142777219","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154","series":"1.2.826.0.1.3680043.8.498.94756993390891286090736114809748154669","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28873180831637763124652074775631752918","series":"1.2.826.0.1.3680043.8.498.12865622144228605745220336814070760602","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28944156754203202831026290463520445489","series":"1.2.826.0.1.3680043.8.498.12640328895604229086633479467769017732","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28951331578236330522646806432827370713","series":"1.2.826.0.1.3680043.8.498.15993446535734977805925764750137300394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29055892482211392922525222851162308510","series":"1.2.826.0.1.3680043.8.498.31107385463189872350406956647345460573","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29227028592609694700827329867343028944","series":"1.2.826.0.1.3680043.8.498.52263141810725736651291764469557977429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586","series":"1.2.826.0.1.3680043.8.498.77128270017224804186595123731382776731","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.29020082251654848193638272552653467980","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.59372628826483277730290444250049716178","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370","series":"1.2.826.0.1.3680043.8.498.89363105086660833837249897054012523274","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464","series":"1.2.826.0.1.3680043.8.498.10942068970649667120765369766810424191","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29735852958813434703969490187047665950","series":"1.2.826.0.1.3680043.8.498.40324130296784411835902723601961009590","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054","series":"1.2.826.0.1.3680043.8.498.33409841341034789616913822442635260773","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350","series":"1.2.826.0.1.3680043.8.498.33987394228379700697524529666589259318","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29943089198208468441240036404506927883","series":"1.2.826.0.1.3680043.8.498.46361537467402968689898293450956533723","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29976611818313906970687136815346573471","series":"1.2.826.0.1.3680043.8.498.13615381895605791518810220403491483883","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30104491808697888642424160917715722238","series":"1.2.826.0.1.3680043.8.498.82809005488122583061074070405955413387","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.12394471865050070118255267538894423043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934","series":"1.2.826.0.1.3680043.8.498.28331075153450469269361066553162166405","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30233578499383083757206211752001832047","series":"1.2.826.0.1.3680043.8.498.11362221487467389265394928109609442563","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600","series":"1.2.826.0.1.3680043.8.498.33409471063014066720897499555508159257","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30282025087399390929963075109487390046","series":"1.2.826.0.1.3680043.8.498.85001092727284184583317110343868937455","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540","series":"1.2.826.0.1.3680043.8.498.37418784542545257274714661627978038997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30435446905709054197096833427672604260","series":"1.2.826.0.1.3680043.8.498.11440023133062596829746957792032700839","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30517381653283520211188050362394799510","series":"1.2.826.0.1.3680043.8.498.42646967651298851799964576371581464468","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622","series":"1.2.826.0.1.3680043.8.498.11537015652797414245712370468647247201","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30662407196303880392459287553277761223","series":"1.2.826.0.1.3680043.8.498.52537238122873942697674876707778513126","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30673411238679998084995204090753297949","series":"1.2.826.0.1.3680043.8.498.12599750580812755805445726215183947350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520","series":"1.2.826.0.1.3680043.8.498.20643162464420248234443285354937591898","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30800677497901063887487934675082066316","series":"1.2.826.0.1.3680043.8.498.40177475027058425586548942153382208536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30836611607735372299752226690388090391","series":"1.2.826.0.1.3680043.8.498.12862742486414290690859287032222106454","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425","series":"1.2.826.0.1.3680043.8.498.62049710655335968997316093679389232040","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066","series":"1.2.826.0.1.3680043.8.498.57058083148303659570170144172751657918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961","series":"1.2.826.0.1.3680043.8.498.11449841620369646929510224830085679244","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31082592133742929889611857821843859701","series":"1.2.826.0.1.3680043.8.498.12840294273863237810221963068776394646","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31145453746697192222724674918300432873","series":"1.2.826.0.1.3680043.8.498.10711254052105074090829150881104336555","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777","series":"1.2.826.0.1.3680043.8.498.98404626354935268302206096840188572939","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31218039240665208717816408042644491715","series":"1.2.826.0.1.3680043.8.498.43960999255854690261151135338733914263","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119","series":"1.2.826.0.1.3680043.8.498.26662234509891877914463030709445295785","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31284572943302383501947308874558551301","series":"1.2.826.0.1.3680043.8.498.22721809464579768144645937539606936320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486","series":"1.2.826.0.1.3680043.8.498.10310157687254813702055044547334806827","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31482725509752510642857072190147133461","series":"1.2.826.0.1.3680043.8.498.76500720410432644256339456065384214359","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31610133883321894510649078615659708431","series":"1.2.826.0.1.3680043.8.498.11136418313598932078462434388195471321","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31745644522310786426860265012373331979","series":"1.2.826.0.1.3680043.8.498.12086938513521288747785619456889072825","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721","series":"1.2.826.0.1.3680043.8.498.39467621018081305120281593306536546281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31842732064173292828619477783097732181","series":"1.2.826.0.1.3680043.8.498.59238630819367838885162493854580332614","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100","series":"1.2.826.0.1.3680043.8.498.11135750104148915686150411007241606201","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778","series":"1.2.826.0.1.3680043.8.498.20937486053640043930777588283181489260","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32109377199577933060124357239881959602","series":"1.2.826.0.1.3680043.8.498.34414670879496968255416215516499792274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32179217289301203414651193280580594771","series":"1.2.826.0.1.3680043.8.498.52983608450618214560553745209921760616","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795","series":"1.2.826.0.1.3680043.8.498.12223082435000542901229785770750937529","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945","series":"1.2.826.0.1.3680043.8.498.12475417103671366572286789031735364146","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32329374180424300100441619205766536205","series":"1.2.826.0.1.3680043.8.498.11395584000470322418630609127980249496","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007","series":"1.2.826.0.1.3680043.8.498.61713826343542661159205235588811514393","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32501049887140121289143479773995481982","series":"1.2.826.0.1.3680043.8.498.12765634174351563911172688665204565300","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32521102053110469817073126425955562934","series":"1.2.826.0.1.3680043.8.498.29824423478408356020241375471503273100","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664","series":"1.2.826.0.1.3680043.8.498.10613155904209636655846066222528011126","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.11479498991501258480514559974879681783","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677","series":"1.2.826.0.1.3680043.8.498.12801651967471924955468666420678250639","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32784816158214124982843838883515640569","series":"1.2.826.0.1.3680043.8.498.11580597474172807872899738816585565542","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133","series":"1.2.826.0.1.3680043.8.498.39764856440440663129337735665030597357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569","series":"1.2.826.0.1.3680043.8.498.69232396493218547991107699939319200492","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534","series":"1.2.826.0.1.3680043.8.498.93911776307061502092499555763151518135","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807","series":"1.2.826.0.1.3680043.8.498.17078169896297490132569300234102419887","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33284228284209148349668722931307588402","series":"1.2.826.0.1.3680043.8.498.11929605906760438192078258534215181752","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","series":"1.2.826.0.1.3680043.8.498.56863789870500825059022019273485615376","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086","series":"1.2.826.0.1.3680043.8.498.33248264445327799857052124040510150393","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33432461096351771607862098991677662209","series":"1.2.826.0.1.3680043.8.498.79084916623782543795304080212854844123","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314","series":"1.2.826.0.1.3680043.8.498.12637731034664044238288408027102374359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941","series":"1.2.826.0.1.3680043.8.498.12883999866503742498230235501754494715","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33709759248055631658322056320849494088","series":"1.2.826.0.1.3680043.8.498.77269786049211763934643141515950935289","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33841907095190352493353075864604147687","series":"1.2.826.0.1.3680043.8.498.14612996830856650362088420250080877896","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830","series":"1.2.826.0.1.3680043.8.498.95869473252064836239317789672332426095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33957165685646407808982253731894148028","series":"1.2.826.0.1.3680043.8.498.50972566179551528617783302310061436704","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34002185870196144582348922158474418674","series":"1.2.826.0.1.3680043.8.498.65325891784739748770376110257520791257","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.59194259826918785993593154417564626213","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34109091435857467600499949506940613037","series":"1.2.826.0.1.3680043.8.498.50137498973842924730145522944480524765","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34206043272866138622475050809785944941","series":"1.2.826.0.1.3680043.8.498.11091661603319742987606631497522385371","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984","series":"1.2.826.0.1.3680043.8.498.71914319615102084611516046382136254230","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34404554752241524210538769325381023947","series":"1.2.826.0.1.3680043.8.498.11604481494563862006300199863462296259","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34441662343660776323013762699640240389","series":"1.2.826.0.1.3680043.8.498.45094184501007874045048903419545282736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34501976934829436985900782482912019868","series":"1.2.826.0.1.3680043.8.498.11955691145509128772246750970400568757","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.11611209382647481793503209866576787588","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.53666657856688623647372446422045958397","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455","series":"1.2.826.0.1.3680043.8.498.50938947752533500979747897270386543391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34753112960188810416088070814158959510","series":"1.2.826.0.1.3680043.8.498.11285934307627935158460179049649137614","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34795373994453585401035013232069587413","series":"1.2.826.0.1.3680043.8.498.93548718829246801600094774962378621850","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34858782760614383566891001610985452165","series":"1.2.826.0.1.3680043.8.498.73601668113829278968486786042457459718","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.26190161195183929804521180373336032297","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.10715117184099832306581872804705570112","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139","series":"1.2.826.0.1.3680043.8.498.85656718433794968801546551464904770145","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35127678462764233938878060108411012978","series":"1.2.826.0.1.3680043.8.498.84116348024307228093933082111468468645","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841","series":"1.2.826.0.1.3680043.8.498.13210622960158979213494101435774080486","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35273590880523815297008790288567354883","series":"1.2.826.0.1.3680043.8.498.35973016643010062421558985602080991371","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35338198410583984337908932327787063577","series":"1.2.826.0.1.3680043.8.498.16953619970177145296516943097889514274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.13127904396756695943180814076874936581","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35633121799529901420381085923067525212","series":"1.2.826.0.1.3680043.8.498.33692677204812643572456293236780345165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35677515667357567818845364400903668190","series":"1.2.826.0.1.3680043.8.498.26140159716856320330607449529107684224","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777","series":"1.2.826.0.1.3680043.8.498.34244579294687755435313096359765156582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.22291853273059584228244582877183628082","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403","series":"1.2.826.0.1.3680043.8.498.56983001858921358322058403964910466145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278","series":"1.2.826.0.1.3680043.8.498.10123261573053632466615236207077910155","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35906584224533966870202042352292878668","series":"1.2.826.0.1.3680043.8.498.99170385417805295768906328587018931073","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734","series":"1.2.826.0.1.3680043.8.498.12482749429157323937298123201048730465","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.12379043873372197095602717985636151057","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35965770865730278787229040370963322662","series":"1.2.826.0.1.3680043.8.498.15755898326727804563960998133984841638","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36013620117872095227314165026090752632","series":"1.2.826.0.1.3680043.8.498.95320290144269362805560571936789993019","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565","series":"1.2.826.0.1.3680043.8.498.71440054374202727041480230214267282997","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36158956431673597472934120620368656246","series":"1.2.826.0.1.3680043.8.498.44025689666800738831884297136768964066","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976","series":"1.2.826.0.1.3680043.8.498.20040515401122542269172372105791310390","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198","series":"1.2.826.0.1.3680043.8.498.11634209439554650241069791647010222514","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167","series":"1.2.826.0.1.3680043.8.498.55479626162951683760214963011743032789","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300","series":"1.2.826.0.1.3680043.8.498.11633917809434730608601837128199551715","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36521162836904095186654423648490092035","series":"1.2.826.0.1.3680043.8.498.11062891739421655203119402405090469704","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36645945703275956149107297016813148264","series":"1.2.826.0.1.3680043.8.498.11790638330140457038841288513118690998","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374","series":"1.2.826.0.1.3680043.8.498.70834394669026677500079413874943277621","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.12172964188106376339752363126935670613","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36825274340853796392762195174336452919","series":"1.2.826.0.1.3680043.8.498.25172973594300135618885849858871523496","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36891467514776207809384804066253854193","series":"1.2.826.0.1.3680043.8.498.12614022091570838129727813519177489368","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047","series":"1.2.826.0.1.3680043.8.498.77969670428922873451321404422400060461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37005775889857426034942587344620888398","series":"1.2.826.0.1.3680043.8.498.61783827421706880411064356085108580479","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742","series":"1.2.826.0.1.3680043.8.498.22306721889799850550972691123555559051","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727","series":"1.2.826.0.1.3680043.8.498.39060003989991136459403351584839828037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37210322466522537523056852160944888029","series":"1.2.826.0.1.3680043.8.498.14931636789351786845349350605376971100","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316","series":"1.2.826.0.1.3680043.8.498.50657211886020353018125514602375824684","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37371365833775442014007234936409739360","series":"1.2.826.0.1.3680043.8.498.64776763633609723157450839270054192544","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683","series":"1.2.826.0.1.3680043.8.498.13144206930098313641221403308323266222","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37432328401993499374675847888297416449","series":"1.2.826.0.1.3680043.8.498.77924256088090330220927722232069224230","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849","series":"1.2.826.0.1.3680043.8.498.12531567183344017925276037763478730712","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37508036912473928281276902883575142611","series":"1.2.826.0.1.3680043.8.498.68776319947325238011853459005424466886","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37519350121906786941281770770693979271","series":"1.2.826.0.1.3680043.8.498.12785197884776726926348916277245865448","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37580209294522316445963449455348371815","series":"1.2.826.0.1.3680043.8.498.12759787525718393850870713547222477908","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37591590536870291237852764995372956522","series":"1.2.826.0.1.3680043.8.498.53624462120992995279922653231425163336","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37644754953609976838298110418046213654","series":"1.2.826.0.1.3680043.8.498.66315424724605748154875745886012462186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.13090249907355796199513887560916584250","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37838990693631869129590283596686992710","series":"1.2.826.0.1.3680043.8.498.56828052296050332460547311050168487088","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122","series":"1.2.826.0.1.3680043.8.498.10842618279013269966032028256498332413","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335","series":"1.2.826.0.1.3680043.8.498.99233838084734019904393120535239629693","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37969790195791584440906553207150666923","series":"1.2.826.0.1.3680043.8.498.16147071203026322522870552518223576653","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38013372055010421273425628837282557767","series":"1.2.826.0.1.3680043.8.498.43763396019809316543214517648098440738","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221","series":"1.2.826.0.1.3680043.8.498.35335267116242346566320867275264422587","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38095098835657518803781210111463995296","series":"1.2.826.0.1.3680043.8.498.35597939119333606782466423936437502997","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38224750236138877853888930317337541394","series":"1.2.826.0.1.3680043.8.498.31817400782751859452246194369956406439","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38238963443939714344730201261512911003","series":"1.2.826.0.1.3680043.8.498.11244886662814804700681630029268599359","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262","series":"1.2.826.0.1.3680043.8.498.76053789476863984139652252170212154656","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38325494489236992123219618004392575979","series":"1.2.826.0.1.3680043.8.498.34458013667678984862347585998167554361","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552","series":"1.2.826.0.1.3680043.8.498.11102612150853431038852466897191813553","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059","series":"1.2.826.0.1.3680043.8.498.51892973481955667415124624694326584707","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38457315909459574570206119213798163082","series":"1.2.826.0.1.3680043.8.498.41403167635496499904282258601874548576","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38518069950656326194016175319535174299","series":"1.2.826.0.1.3680043.8.498.68936290868349129057794432814051612684","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38567342184301320850258918666303107813","series":"1.2.826.0.1.3680043.8.498.25708785360583926157385762513816263533","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38591915802992557518846862828618375715","series":"1.2.826.0.1.3680043.8.498.34219717417201853114137731199206568304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926","series":"1.2.826.0.1.3680043.8.498.12790581669843529161157907220673572010","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38701927697650476092773702704871872598","series":"1.2.826.0.1.3680043.8.498.12625607435487189592475654114594976813","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604","series":"1.2.826.0.1.3680043.8.498.11173629806557378105678420221771304536","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893","series":"1.2.826.0.1.3680043.8.498.83167623516570314932668566341847216259","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38862475907405766248096419059100131863","series":"1.2.826.0.1.3680043.8.498.10502516332832957171731605223848275608","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405","series":"1.2.826.0.1.3680043.8.498.86160979502276721018176488352074962680","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373","series":"1.2.826.0.1.3680043.8.498.75510988583333855058387330935562864846","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38935694477580522005505074220300062204","series":"1.2.826.0.1.3680043.8.498.32056439928066648097800993660491136863","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799","series":"1.2.826.0.1.3680043.8.498.86539736423952433190800766576827196405","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161","series":"1.2.826.0.1.3680043.8.498.54231242265283192098203073478271464003","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752","series":"1.2.826.0.1.3680043.8.498.48852048405366095505968503744484118350","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39135770176150551756142329948474103964","series":"1.2.826.0.1.3680043.8.498.93456349908692444846137744536468187578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39250844287482885227700969308333354206","series":"1.2.826.0.1.3680043.8.498.28134914265091025899724807663469852924","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39275838957651450909114440721012584183","series":"1.2.826.0.1.3680043.8.498.87423667004286956370813410258114526478","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39394842976048577715164357752399606924","series":"1.2.826.0.1.3680043.8.498.85395963605177149498931382957848778440","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402","series":"1.2.826.0.1.3680043.8.498.47949671640090717516778355817114443914","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39547752108509751520306190293259313911","series":"1.2.826.0.1.3680043.8.498.12095294204842532984053650863296581178","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.29698703954224030413107830478087473889","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39787654910948635229369478006027304578","series":"1.2.826.0.1.3680043.8.498.13940987205136128663179844494029248051","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39845300426582562781355154625767485381","series":"1.2.826.0.1.3680043.8.498.62917574242250118935578549885185494347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39895705984068714346274689335576595443","series":"1.2.826.0.1.3680043.8.498.91290227196277765442098591017929634536","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39954567501513957418422043089587980092","series":"1.2.826.0.1.3680043.8.498.13200238010651788904456138675416778309","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112","series":"1.2.826.0.1.3680043.8.498.70751336721144731452802210221768091128","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40086005762552976451418579375198597540","series":"1.2.826.0.1.3680043.8.498.23052329427277769445134818111586892950","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40196271329411009311273016603705560443","series":"1.2.826.0.1.3680043.8.498.40263563445092659123577688555231087461","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693","series":"1.2.826.0.1.3680043.8.498.56325699121060778474166396458756831229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40269807401373034066229810558300266830","series":"1.2.826.0.1.3680043.8.498.91522159210291129561393192604878351317","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837","series":"1.2.826.0.1.3680043.8.498.12653167598525660581129722649601810778","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.12224357956587373596834905197001608976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40437305053062892226978469238653825745","series":"1.2.826.0.1.3680043.8.498.10316814245108823565228023892710688176","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40487147942341753806704765355921322161","series":"1.2.826.0.1.3680043.8.498.62291591939051793979230141478136653857","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315","series":"1.2.826.0.1.3680043.8.498.85549233657875509124180208600414244211","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655","series":"1.2.826.0.1.3680043.8.498.99009659675479925820730844991598711125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872","series":"1.2.826.0.1.3680043.8.498.49323656310677224435449817906278031406","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40718200801895968234499223301900799376","series":"1.2.826.0.1.3680043.8.498.21634851387875416519063084073634328001","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40765829900885943055669918091033014654","series":"1.2.826.0.1.3680043.8.498.16509806979951455430654441175964865101","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849","series":"1.2.826.0.1.3680043.8.498.11205889729740063581361160084609447079","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754","series":"1.2.826.0.1.3680043.8.498.98851469407755075764228692225929253557","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40987687016832998820664054439678138878","series":"1.2.826.0.1.3680043.8.498.92432836949442132635493955837889709099","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994","series":"1.2.826.0.1.3680043.8.498.11731349305797712663852103181722870052","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260","series":"1.2.826.0.1.3680043.8.498.89289484860578548018377355663439282994","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41197678541534079475667407055098243437","series":"1.2.826.0.1.3680043.8.498.12192482119169571407426025583299995372","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41315359092354940082611381451934245995","series":"1.2.826.0.1.3680043.8.498.78789314790388000971400069262288769505","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41363979247025384202932112601795525615","series":"1.2.826.0.1.3680043.8.498.92625021151832830158472311078958087631","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.82434297805034574489661243417957940925","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41454762422897656673710145732036115764","series":"1.2.826.0.1.3680043.8.498.13343422264705961073426159010672178259","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41587640208769641326416798167574152049","series":"1.2.826.0.1.3680043.8.498.47158657277873538940752615960387390178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415","series":"1.2.826.0.1.3680043.8.498.11086047179126770658969878757602368527","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41640481150213029810772014344448781554","series":"1.2.826.0.1.3680043.8.498.97330234840707787563422878066979815132","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.70160273359870384482914018667728370583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.74389339348633000283070126504845546392","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.12058426034630231359706906833074811916","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537","series":"1.2.826.0.1.3680043.8.498.15077695905490368820220556791537927481","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004","series":"1.2.826.0.1.3680043.8.498.48476085630252001385679276408660076665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41939824564209194493855981805593285528","series":"1.2.826.0.1.3680043.8.498.11082518646214081292682819015790807211","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42018410640565840140895012243840256810","series":"1.2.826.0.1.3680043.8.498.18270220616449328909362140456946378902","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782","series":"1.2.826.0.1.3680043.8.498.29365786428903046055932031003402998425","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42120645248710860870761681999665860884","series":"1.2.826.0.1.3680043.8.498.62291506216744704718956570387052190418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42162585695624963017992042571207363427","series":"1.2.826.0.1.3680043.8.498.14631238864930521559141343838568823061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947","series":"1.2.826.0.1.3680043.8.498.34342389249712345617355430043158843384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275","series":"1.2.826.0.1.3680043.8.498.23837845959000429243461967134083788003","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936","series":"1.2.826.0.1.3680043.8.498.13219254582952475782916164056619695364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42566296562794128096734153388840524012","series":"1.2.826.0.1.3680043.8.498.88325675991640074216313231142366002280","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42587087495729026068405824960397052997","series":"1.2.826.0.1.3680043.8.498.12075883669431090106973974653757785069","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006","series":"1.2.826.0.1.3680043.8.498.78464354542659878926517236583851057469","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42768603719459834345001093623865167866","series":"1.2.826.0.1.3680043.8.498.88779093891245670998884099602175810203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775","series":"1.2.826.0.1.3680043.8.498.26502583786790739611490700701958346460","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42847376403722341195083350288786052637","series":"1.2.826.0.1.3680043.8.498.67431117873801671756845741621930733305","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42896117106271721157113072562739356381","series":"1.2.826.0.1.3680043.8.498.35185578749137671935279145598286627921","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42965031665568825650795822006244115609","series":"1.2.826.0.1.3680043.8.498.41499378005437031675309516054192907529","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42990542863216044717944418712887800295","series":"1.2.826.0.1.3680043.8.498.45240005126882461974316801792740032103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43065391907984821806263204343792008938","series":"1.2.826.0.1.3680043.8.498.10651249113129471417981166045990476339","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43127139928509033152155587499980767364","series":"1.2.826.0.1.3680043.8.498.12570091551922316577825074236587985059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43256905750043156063416845850825787417","series":"1.2.826.0.1.3680043.8.498.12572819196288470999492444894478411505","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557","series":"1.2.826.0.1.3680043.8.498.12632090277999341255381324148845620469","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43355923025859051425546451466843844346","series":"1.2.826.0.1.3680043.8.498.10496726445508964324123634516946498475","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343","series":"1.2.826.0.1.3680043.8.498.11349497963009707456658729263484052370","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490","series":"1.2.826.0.1.3680043.8.498.46729926145365187392470439519935516383","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43587826365647713044374316837330397375","series":"1.2.826.0.1.3680043.8.498.78106361210783162548462422913099737562","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","series":"1.2.826.0.1.3680043.8.498.18605121067419868568593338093911522696","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43832771919542075229536000564653592465","series":"1.2.826.0.1.3680043.8.498.42818529651136909191339331554344996305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44002376543340544410495269060824194475","series":"1.2.826.0.1.3680043.8.498.14790293743954408126668944088400614393","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208","series":"1.2.826.0.1.3680043.8.498.95748826939469455073629653649627846507","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889","series":"1.2.826.0.1.3680043.8.498.15999410242543743270650580959998554471","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44333089398623004498721836325723633455","series":"1.2.826.0.1.3680043.8.498.42970232616919668476513819310652439882","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44405554266141576380633014711246118209","series":"1.2.826.0.1.3680043.8.498.97452699101798448003772932020106521159","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.54780131475366192858819708791631106146","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44525218699266472141419416162979441275","series":"1.2.826.0.1.3680043.8.498.12875207977589157481166781934940350135","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44654558140536370040701171884970036040","series":"1.2.826.0.1.3680043.8.498.68714611630653526462757519415373492467","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.13994575905448125261001408327748649232","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44792591585917525576461101442761037507","series":"1.2.826.0.1.3680043.8.498.12057149255458531629994172986772705601","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.13019897854430552885545905607891115228","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44902036121220687805715010480088379768","series":"1.2.826.0.1.3680043.8.498.79599539905159498478208902903854710624","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44996883917122941421301655137158093858","series":"1.2.826.0.1.3680043.8.498.16762900805226699036681306330351785964","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45059917085750212913588683242522799083","series":"1.2.826.0.1.3680043.8.498.98841607867643759857905840918379309122","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634","series":"1.2.826.0.1.3680043.8.498.90913860843583326358367260708548832936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45207812477654339877341538582090218293","series":"1.2.826.0.1.3680043.8.498.11870819330895659838485694086349279513","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45286493874121093392171044034263851961","series":"1.2.826.0.1.3680043.8.498.52016288049342865015201485054789779074","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45316453179223822297459860764883030492","series":"1.2.826.0.1.3680043.8.498.80936737767794943904471001854957236517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641","series":"1.2.826.0.1.3680043.8.498.17863998514103268926123081855303025153","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45380312679924593883892686293576260616","series":"1.2.826.0.1.3680043.8.498.40627989332448217448979285245833576882","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45454539422542821586939369440135728144","series":"1.2.826.0.1.3680043.8.498.81388405718442124640655931179827051593","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","series":"1.2.826.0.1.3680043.8.498.39961280345385272129890052548502421415","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45519502487291848581539950159353124045","series":"1.2.826.0.1.3680043.8.498.13226766539418027648368523696684350727","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45644703214235636656820126625301033696","series":"1.2.826.0.1.3680043.8.498.13609737040877810778655932072215057654","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121","series":"1.2.826.0.1.3680043.8.498.67005470245778704341537747576592237982","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426","series":"1.2.826.0.1.3680043.8.498.98101160251079523102871218205162875143","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549","series":"1.2.826.0.1.3680043.8.498.11271850341341224124461762254750898588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663","series":"1.2.826.0.1.3680043.8.498.42210519213192645850547619797319911443","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46051119885630602872050329412008545893","series":"1.2.826.0.1.3680043.8.498.71818139177298912809715777930247053937","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46123761840464615814604226041496278995","series":"1.2.826.0.1.3680043.8.498.72955514055845639212292241028960486576","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46228189366201925238527911905072482095","series":"1.2.826.0.1.3680043.8.498.10836880358008609823233854260377133357","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46268820642949526111807948525363811264","series":"1.2.826.0.1.3680043.8.498.60203354089415349294679279146890763687","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469","series":"1.2.826.0.1.3680043.8.498.19770768429155687460227729937730645445","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46335849264221717416382765091672040855","series":"1.2.826.0.1.3680043.8.498.27360782581991281253727024734780792353","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46452612704507986055186132736451572677","series":"1.2.826.0.1.3680043.8.498.55649724122526771911133043623701184303","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265","series":"1.2.826.0.1.3680043.8.498.85760718884370947963667299188590508378","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46713978512945139851373191422321462143","series":"1.2.826.0.1.3680043.8.498.13234752475889871837062999424929876750","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241","series":"1.2.826.0.1.3680043.8.498.79376558575824485414324592009535389094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46797419946872257347118338518757027995","series":"1.2.826.0.1.3680043.8.498.37345539972964028015406501463318188342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657","series":"1.2.826.0.1.3680043.8.498.54451773913128147533875839402655593037","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46909729013880708074310736136355969956","series":"1.2.826.0.1.3680043.8.498.57782642803493435055953187277929008023","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.10632531192811899276510557763004424886","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037","series":"1.2.826.0.1.3680043.8.498.24081395400527878314935473030792061785","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47094981922544354899183959858078311972","series":"1.2.826.0.1.3680043.8.498.56537160926328122162193063250728009750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47255564202539432049700324486197024855","series":"1.2.826.0.1.3680043.8.498.50202632759097966590981007681607031084","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47268599732083036147289040424392838620","series":"1.2.826.0.1.3680043.8.498.51747954664454310413324813235861297047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953","series":"1.2.826.0.1.3680043.8.498.11248953021770098948491119272815822603","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825","series":"1.2.826.0.1.3680043.8.498.65617286662135969783597937766337935137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47651715105501347255071941051553742767","series":"1.2.826.0.1.3680043.8.498.83456851323553107881270209681965289357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400","series":"1.2.826.0.1.3680043.8.498.18728301867243170650298718200439958539","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959","series":"1.2.826.0.1.3680043.8.498.98479534714482990151722447229269064043","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47855330524708374523226349383799712473","series":"1.2.826.0.1.3680043.8.498.61803824255873110166848023759356747224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187","series":"1.2.826.0.1.3680043.8.498.47044501123729808256032287793135859318","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835","series":"1.2.826.0.1.3680043.8.498.53423098088174479480562356174820339737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331","series":"1.2.826.0.1.3680043.8.498.37101433006786027034662990659272782561","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48308485987218296338173512060246238691","series":"1.2.826.0.1.3680043.8.498.11035903844741280695764529240455336555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48368929760320103375578806071106317953","series":"1.2.826.0.1.3680043.8.498.73592682936657880228906962611354194380","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799","series":"1.2.826.0.1.3680043.8.498.73912540170284362404124353868122795525","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48521453765449688946535921546652765399","series":"1.2.826.0.1.3680043.8.498.74120296037141566010690143984918875383","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","series":"1.2.826.0.1.3680043.8.498.37811684258662085729606651071265289942","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48750912584073496502856009873745362236","series":"1.2.826.0.1.3680043.8.498.10527372249218195962305749228173275945","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48780598610511011488578963416186905560","series":"1.2.826.0.1.3680043.8.498.12422818164319649865913946366507108072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48829151552094424673693975665028690142","series":"1.2.826.0.1.3680043.8.498.70557753781294992527168673138869192602","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48872083192774100826667097552356058756","series":"1.2.826.0.1.3680043.8.498.68089489344636251286485750844209246770","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132","series":"1.2.826.0.1.3680043.8.498.31457592019990349918394410240358497641","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547","series":"1.2.826.0.1.3680043.8.498.51163198769833146774064032099004802438","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49054454369199540406654153762248931799","series":"1.2.826.0.1.3680043.8.498.81146997407467703477664111691209737756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49163335688297423808344168197551387860","series":"1.2.826.0.1.3680043.8.498.10679748782668302982495943645739668758","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49219586198038249030222134813685536295","series":"1.2.826.0.1.3680043.8.498.96468768646607484664746598872174996245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.92669247109314784934272630802927647138","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305","series":"1.2.826.0.1.3680043.8.498.13030619801715782427296805015936762203","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49492260217998945587168161555623476238","series":"1.2.826.0.1.3680043.8.498.79450289063467219781887581880284242613","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49500752658775616590911569693817273205","series":"1.2.826.0.1.3680043.8.498.34753184751952551117163199540050522763","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654","series":"1.2.826.0.1.3680043.8.498.93190060466488991406616690595324698010","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.10095976511552480616263531780754388395","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921","series":"1.2.826.0.1.3680043.8.498.44352556073252636373656389476736278822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912","series":"1.2.826.0.1.3680043.8.498.13381034830455665040012587130294895377","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125","series":"1.2.826.0.1.3680043.8.498.13484361742948284380868343927253307277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.26566443917372254382268727900398751892","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49984662916977328110175445895515541267","series":"1.2.826.0.1.3680043.8.498.12798675252584650703536924517644614633","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.96711627810353112042032162256642799723","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50107479256735739327080544431929221135","series":"1.2.826.0.1.3680043.8.498.96533657812606570868342312631890868618","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50182378358892377789975570268512056553","series":"1.2.826.0.1.3680043.8.498.11115808148933420799804983642041670734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50220991646207533313828550949660577504","series":"1.2.826.0.1.3680043.8.498.86207365097357493058682516350246714320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50259348432783983788186173025555513832","series":"1.2.826.0.1.3680043.8.498.70001413971808714603403599096046928564","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50318760104506317341991314680640521601","series":"1.2.826.0.1.3680043.8.498.62136697980449360150277862870014855871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50392110746914984873009735588879049321","series":"1.2.826.0.1.3680043.8.498.10961393130652586253371912064444508918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.12329731090121469197061872962790358505","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550","series":"1.2.826.0.1.3680043.8.498.11237772545849268522155980511201459535","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50648934951234229265309905009982297515","series":"1.2.826.0.1.3680043.8.498.73286770990821860864139138959099697345","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50667444467788034101157821576803778774","series":"1.2.826.0.1.3680043.8.498.11824830597690701402680193384703293894","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679","series":"1.2.826.0.1.3680043.8.498.67970644816420189411804849779708083793","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241","series":"1.2.826.0.1.3680043.8.498.13339143510757473604428481078441579931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652","series":"1.2.826.0.1.3680043.8.498.23933285111262543725299380433815269852","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50819987120895374623148160166392914169","series":"1.2.826.0.1.3680043.8.498.15279586776593190612307274618431607764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50844227724977857739867646104945090414","series":"1.2.826.0.1.3680043.8.498.73899431387459813614219417486869494599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50882628969985977810341532747520504841","series":"1.2.826.0.1.3680043.8.498.16206651435564591742003363880445758684","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50980137249034338585131194965993785201","series":"1.2.826.0.1.3680043.8.498.89501584879748058341341081665355537203","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51054592064718900209123281525185013961","series":"1.2.826.0.1.3680043.8.498.64468655693993966829414080337540150526","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053","series":"1.2.826.0.1.3680043.8.498.58506750828096451884073325917270568150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.12012759821275689500338805326342042736","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51379389400506715529323532959275254794","series":"1.2.826.0.1.3680043.8.498.66739307797809153690395636662406502040","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51427159850769972151411336025891779423","series":"1.2.826.0.1.3680043.8.498.63094491113418248873423400495222079906","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51509214978951980397275258374070843967","series":"1.2.826.0.1.3680043.8.498.23510131396133648721219455894463817812","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51676679394093588751371807453793655464","series":"1.2.826.0.1.3680043.8.498.14234754517502799867709551562647219748","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131","series":"1.2.826.0.1.3680043.8.498.27109511300870513478599334416793902253","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099","series":"1.2.826.0.1.3680043.8.498.72711228875692871314457148413494580235","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.50880585216242029849414147924857258221","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51831304809871808790657397599860793132","series":"1.2.826.0.1.3680043.8.498.68408272106519202566600855372632214816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51886877282987244004912794726043055730","series":"1.2.826.0.1.3680043.8.498.33869099175995049100681399990650833627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215","series":"1.2.826.0.1.3680043.8.498.10136706417482729342220705937754879875","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51985313438970133264846259111827877590","series":"1.2.826.0.1.3680043.8.498.66245801926566907892514001593647015931","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329","series":"1.2.826.0.1.3680043.8.498.21030987249142742029463209824995574560","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52067134682915759489080529708019359378","series":"1.2.826.0.1.3680043.8.498.56877968062075299025351127723187730431","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52110163572817528199125320120779141456","series":"1.2.826.0.1.3680043.8.498.10807553589733085065496055308665684139","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.29529069216265949176042037213217299580","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52330656126848153803758928167944040055","series":"1.2.826.0.1.3680043.8.498.42466334600991973904244828114141433819","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52409113683924904563676007800283978062","series":"1.2.826.0.1.3680043.8.498.17421345569289323731938642718299526117","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52437948508356178175558637950043479411","series":"1.2.826.0.1.3680043.8.498.57811292686362748995743178962515742301","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52475562989926669452280424331791927661","series":"1.2.826.0.1.3680043.8.498.98897557057659719818478350068902205806","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52558822670236188507569589448789513993","series":"1.2.826.0.1.3680043.8.498.13241487131784990193048067004232873404","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650","series":"1.2.826.0.1.3680043.8.498.34984178294419943415265851424098339860","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52785983814702054185130109942936184688","series":"1.2.826.0.1.3680043.8.498.12352280451371900578390254652406558764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52846236545327527967419125989598675101","series":"1.2.826.0.1.3680043.8.498.69647589636205866034368210153911169875","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52985748674486376309978868177895755694","series":"1.2.826.0.1.3680043.8.498.48576923416027321532605135558660962155","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53076472372145933727990308381815269368","series":"1.2.826.0.1.3680043.8.498.12906896700649829606733971264612665674","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711","series":"1.2.826.0.1.3680043.8.498.38853658922106222591683088171952613872","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53131430415327479511358970615140982847","series":"1.2.826.0.1.3680043.8.498.83650928046348381589047461742325185265","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53156118232564605912649705207438612782","series":"1.2.826.0.1.3680043.8.498.13143288708576903121337331136279204555","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53221818460567216467597044606040911702","series":"1.2.826.0.1.3680043.8.498.11099103295422233393190914225997596328","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133","series":"1.2.826.0.1.3680043.8.498.74491703286119514590952118398217081838","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501","series":"1.2.826.0.1.3680043.8.498.11590718664561192294686134361779269726","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53361558447470435467999344560545063264","series":"1.2.826.0.1.3680043.8.498.13157258277738941835157388553110862312","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643","series":"1.2.826.0.1.3680043.8.498.11838149764282938564694341467168969847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53528011450161116437223332448942321596","series":"1.2.826.0.1.3680043.8.498.73670800841106517459705649137434571264","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53628897373416863107623485630264501760","series":"1.2.826.0.1.3680043.8.498.14540854081092695486250662819028313481","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027","series":"1.2.826.0.1.3680043.8.498.67937431632108140573066239225023132853","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53797236075631231042400235117445767744","series":"1.2.826.0.1.3680043.8.498.11499293383042031709396209112360967435","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965","series":"1.2.826.0.1.3680043.8.498.23912097805930675658873723815830502909","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53968943793571323467397128102531786394","series":"1.2.826.0.1.3680043.8.498.10424403934395193990802901266410553544","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54049224487950229496345484554581532711","series":"1.2.826.0.1.3680043.8.498.21268202916896988215408100827712867619","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54100786607660749952814251186461344633","series":"1.2.826.0.1.3680043.8.498.27535556969074131662602286712164913788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54138747504377266740838999528964336002","series":"1.2.826.0.1.3680043.8.498.63934134101685014275733581384570844854","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54202117421897784469771192940597638508","series":"1.2.826.0.1.3680043.8.498.53455839174072776163681305028067000420","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043","series":"1.2.826.0.1.3680043.8.498.64593554904620412219490534026988957490","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54316764984549863227549326352573342232","series":"1.2.826.0.1.3680043.8.498.82060053410905304762823187190718293346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54394066317295451515247748115888976860","series":"1.2.826.0.1.3680043.8.498.59544976131805817518979534872249462489","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54451515695681765924346479727204451807","series":"1.2.826.0.1.3680043.8.498.86057296802772559342555113863651146543","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54483132338261866032495309977418409299","series":"1.2.826.0.1.3680043.8.498.70326407748299885487865305666504661063","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54589880111295324708950851057241927191","series":"1.2.826.0.1.3680043.8.498.70398704453593468773940773371820431926","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54662745951809363774369202018332309887","series":"1.2.826.0.1.3680043.8.498.35897100654898360088987202970778248545","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642","series":"1.2.826.0.1.3680043.8.498.26318909352232978055329845423922399755","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54711709751232783991855540573189433446","series":"1.2.826.0.1.3680043.8.498.43018747300328336415483209485280828167","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54761306437713414745748829128854749469","series":"1.2.826.0.1.3680043.8.498.10271572304980795272630404717704617248","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54866613282412053616997257524944654864","series":"1.2.826.0.1.3680043.8.498.11315571178408305775702216070139881476","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55008913229055942530957507434366806883","series":"1.2.826.0.1.3680043.8.498.98610684208637789220593581007706033851","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55061039227197546099231336400032878562","series":"1.2.826.0.1.3680043.8.498.67757443572175729313602937741720555825","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.36739170534252006647987034699328018511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55202459816190607876303419067810637245","series":"1.2.826.0.1.3680043.8.498.72213256200317885472156633629686003417","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763","series":"1.2.826.0.1.3680043.8.498.12026086828420700139629793624876033332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55346033326903331672579723063380636237","series":"1.2.826.0.1.3680043.8.498.75218311873618147859586293371013395795","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246","series":"1.2.826.0.1.3680043.8.498.52366412781464625736946031904674893891","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870","series":"1.2.826.0.1.3680043.8.498.64245558369175615284996172022556450489","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55687452977471862155042703116192295932","series":"1.2.826.0.1.3680043.8.498.69970939909242676583180874885149695875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.76792245602347626714433416064275398934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55808772152667135571939417929724526296","series":"1.2.826.0.1.3680043.8.498.12297339884975255305970162246993902007","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55856308312040330141050707531982127791","series":"1.2.826.0.1.3680043.8.498.11202352712535513595772094333774671586","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285","series":"1.2.826.0.1.3680043.8.498.72940860181582718709406371188118426271","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55971152627433130850649668964887611410","series":"1.2.826.0.1.3680043.8.498.12654225107689606518556895204767045619","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.35101531992558756682156916285996043350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56054334296747411394367010227992359191","series":"1.2.826.0.1.3680043.8.498.65330799781741084078518744983780279839","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56091473614612257999053529249190958828","series":"1.2.826.0.1.3680043.8.498.81407765259791515212216812108481715631","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","series":"1.2.826.0.1.3680043.8.498.12126224685184671583598971674923776595","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56216873861595017403772627709751840588","series":"1.2.826.0.1.3680043.8.498.59845525838309370490882224194835453551","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56305445410093481072159375249067050124","series":"1.2.826.0.1.3680043.8.498.11881613930029235152037543901492687999","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770","series":"1.2.826.0.1.3680043.8.498.13288978200965881058758545848262205769","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56506415277488978301157490514745400410","series":"1.2.826.0.1.3680043.8.498.28408613348127900994178136407542084256","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.11378430477818803105723310530334159775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56628956454784173189908113425348977923","series":"1.2.826.0.1.3680043.8.498.13285814973715961185003416880734307840","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879","series":"1.2.826.0.1.3680043.8.498.34767870080509475730797549235681654208","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56786556618312090924315296993756843070","series":"1.2.826.0.1.3680043.8.498.12046341483041368351521839685471202554","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56798070549054124854685511360777559033","series":"1.2.826.0.1.3680043.8.498.50143747924391806772136799943283820319","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883","series":"1.2.826.0.1.3680043.8.498.33439491340839385485058832670746405709","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068","series":"1.2.826.0.1.3680043.8.498.11566271729318380433790148930971731541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56939860329243258146102948466697018543","series":"1.2.826.0.1.3680043.8.498.70402794437195927963796877068018680457","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754","series":"1.2.826.0.1.3680043.8.498.11641255858027431006383452562709784270","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56995522547527264862064392266001620445","series":"1.2.826.0.1.3680043.8.498.21920494606503334363219909611020040188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57074072196714325300896734167161517369","series":"1.2.826.0.1.3680043.8.498.10735810284589727332383392586760990822","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852","series":"1.2.826.0.1.3680043.8.498.26939775043581312518725940614338364937","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57316522861709598765962486949846744486","series":"1.2.826.0.1.3680043.8.498.12585051133550378068144266134381071381","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57406640473235989459885313797729459197","series":"1.2.826.0.1.3680043.8.498.10966685481951482837449712494474293815","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57537915810250993237120953960404180449","series":"1.2.826.0.1.3680043.8.498.13382502296658042741430850757352860136","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044","series":"1.2.826.0.1.3680043.8.498.12854952550856853550340362116661084050","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.58996715070760154170165789158013415429","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57829173797358025677240411871600928097","series":"1.2.826.0.1.3680043.8.498.11111973263568543531659479781790545384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57845032486422442404222046144765204905","series":"1.2.826.0.1.3680043.8.498.88678952986475668148840226745717484665","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.12993161009741911762118825521333327673","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.31318867085292978389188025630228256214","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887","series":"1.2.826.0.1.3680043.8.498.13254590511368053923505203806464809205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58261214953729037661916565473101636690","series":"1.2.826.0.1.3680043.8.498.54842398862611895870615528418169416187","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195","series":"1.2.826.0.1.3680043.8.498.72273298187048984316942328821636254199","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58366682083795592678613986311570432419","series":"1.2.826.0.1.3680043.8.498.51407765261734109803107246259349234179","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292","series":"1.2.826.0.1.3680043.8.498.10615683486182529970607550369175489525","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58482152742263188503728990365891896621","series":"1.2.826.0.1.3680043.8.498.77039372287525479726991192903401870475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58522891982092784622515590012748600644","series":"1.2.826.0.1.3680043.8.498.41572010764905387486695812564918888245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434","series":"1.2.826.0.1.3680043.8.498.82017491166711395173161427403216048751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058","series":"1.2.826.0.1.3680043.8.498.96677480184789939253990775854822773183","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485","series":"1.2.826.0.1.3680043.8.498.18177339049964325318744851095416359508","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744","series":"1.2.826.0.1.3680043.8.498.13608514272649787035088891518891540052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.40191252071123629741249841467212597795","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.57012793738440425781659432525375709673","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.11262411996889513520881982659041442406","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.12862267767945475781225056421142139823","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.12675966507276604258707757221878488210","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769","series":"1.2.826.0.1.3680043.8.498.46272351658477072078382588140627690113","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59140342594072117638979402278277248612","series":"1.2.826.0.1.3680043.8.498.11457693266539611427792866935928897350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59309239681970150083311665704264518844","series":"1.2.826.0.1.3680043.8.498.57175877152407846944571624687393809630","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59426134288436219215525445839849588348","series":"1.2.826.0.1.3680043.8.498.10160709918251093723402612484866267060","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59475691595504590121311599301170361869","series":"1.2.826.0.1.3680043.8.498.10830810283963940523650667026727982959","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59511171909586924989766685687032462369","series":"1.2.826.0.1.3680043.8.498.13133952899340381321632768003438303368","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.11592784420073891932839288792087525494","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235","series":"1.2.826.0.1.3680043.8.498.11657077814499231237084469827672939806","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59826277112503440743678968423276737254","series":"1.2.826.0.1.3680043.8.498.29649291812959207908791576259048957894","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59864697179957009967756900038846073439","series":"1.2.826.0.1.3680043.8.498.61884933630516822310593153104675128503","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.12556941192548691519972162168414423518","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59939977812549570931951749553467050226","series":"1.2.826.0.1.3680043.8.498.14391911196181091117953811386967857735","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59973351395253648445741322652773810641","series":"1.2.826.0.1.3680043.8.498.10149781463386680847244289239021332431","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357","series":"1.2.826.0.1.3680043.8.498.11299061142851328809988486741512571322","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162","series":"1.2.826.0.1.3680043.8.498.10278857190604912362726848501313261316","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.87733321952939078051555338423757271108","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60339775486026531260673620094004244044","series":"1.2.826.0.1.3680043.8.498.12422709895393720912365220719469647266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60430475172382878092821699001847797153","series":"1.2.826.0.1.3680043.8.498.52500147450621079864923290371305374442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184","series":"1.2.826.0.1.3680043.8.498.50160626260369745093067696533557307917","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60473167624534682552272469123806503959","series":"1.2.826.0.1.3680043.8.498.75542062349533027055002922019713958586","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60492772672040498730937577391401159109","series":"1.2.826.0.1.3680043.8.498.51872633124601700339490947016949780091","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474","series":"1.2.826.0.1.3680043.8.498.13067497764835544424551206486030882779","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60633008040453481523409852849000823589","series":"1.2.826.0.1.3680043.8.498.14153834994663914003467513220123012620","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60723378470101669902158229125336455520","series":"1.2.826.0.1.3680043.8.498.12895628426521618643656578239031759540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592","series":"1.2.826.0.1.3680043.8.498.11716692397603814589105338507768118071","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60965286164497194740320124904525623873","series":"1.2.826.0.1.3680043.8.498.46646668038861367920478478967478379416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61072566792285434404885416160728089641","series":"1.2.826.0.1.3680043.8.498.16090986297850663037723926062026107577","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106","series":"1.2.826.0.1.3680043.8.498.32092039368235372072880132217272291590","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841","series":"1.2.826.0.1.3680043.8.498.22730006317809985769212127610977277379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","series":"1.2.826.0.1.3680043.8.498.95484520744478859085362227441435534232","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679","series":"1.2.826.0.1.3680043.8.498.11902283377083379861809778402783152734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61456481310419978302697858056095086811","series":"1.2.826.0.1.3680043.8.498.73216875745600090396583622788253264039","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.26607829166183712241249013494031931067","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417","series":"1.2.826.0.1.3680043.8.498.12501562571532694155685548190054074220","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61630894340791204767579628554826500984","series":"1.2.826.0.1.3680043.8.498.82062302399476283509778492168712869367","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61812343111344551704536863711676179812","series":"1.2.826.0.1.3680043.8.498.12379472943483013736916548364569556022","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61904856234837992279531381374350177464","series":"1.2.826.0.1.3680043.8.498.97227983694932612085666776733447825783","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61981891095235024266625598999854298547","series":"1.2.826.0.1.3680043.8.498.41199505393766194719625323086802693057","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.10160577120764588899722334088776455615","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.10159181737836996828636807284184166678","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62217447760619326492634156203608889396","series":"1.2.826.0.1.3680043.8.498.63108740519630576316996064854245532704","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.51270451167641406073939343777167797229","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62383135836496954297324866452887364097","series":"1.2.826.0.1.3680043.8.498.91452057103604152458114019177417522019","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62406588098851994968957074186291720431","series":"1.2.826.0.1.3680043.8.498.50060311516243696357852377656073148395","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272","series":"1.2.826.0.1.3680043.8.498.13135725933598764937203146608237987691","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62662430565901539554882496013173225758","series":"1.2.826.0.1.3680043.8.498.83504190518936729341819941277949674579","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62752755711205966800746994120425231855","series":"1.2.826.0.1.3680043.8.498.43522616608569126042739376397513507279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62782163204193388751869649423542931657","series":"1.2.826.0.1.3680043.8.498.50078944801330739456316118675034862770","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103","series":"1.2.826.0.1.3680043.8.498.11207871609430313446729722081236557658","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62948920639798990757395955917890550512","series":"1.2.826.0.1.3680043.8.498.76036537079550376077806038567985151417","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63005973438879239920248535292375481294","series":"1.2.826.0.1.3680043.8.498.31442426517656269627013507219740039278","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.91058825215190683163362418222250170314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410","series":"1.2.826.0.1.3680043.8.498.68377837328795484211635001095304804162","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63195088465290711580668024587233364997","series":"1.2.826.0.1.3680043.8.498.45590694205636119761338630378476928391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63204735160524958513470115761010950181","series":"1.2.826.0.1.3680043.8.498.70390963942313053085444371858429659553","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.44938304032865630998945366468919467307","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540","series":"1.2.826.0.1.3680043.8.498.49140645125937418771554053281329463400","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63360204391136938955220146755331421179","series":"1.2.826.0.1.3680043.8.498.41400938796484859656333629660898231858","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63424633611847022112809173449690688015","series":"1.2.826.0.1.3680043.8.498.39724352224865987155260818969890829035","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.12049077392556854325897855218285590670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343","series":"1.2.826.0.1.3680043.8.498.38535917330460769346947304478781299462","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63539809431911516912857504094611479480","series":"1.2.826.0.1.3680043.8.498.16753318058331431959100812015720085446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941","series":"1.2.826.0.1.3680043.8.498.10108190312210636974463078448762808390","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63584266247144369196126771996126173497","series":"1.2.826.0.1.3680043.8.498.38189490480936970665362599953400622017","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63726512375511246975989322174096818969","series":"1.2.826.0.1.3680043.8.498.34104483831584436388103003018723280226","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63849051301265505051181245915656194652","series":"1.2.826.0.1.3680043.8.498.33497694482519754115994402651829373658","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63986652686300452077410299429386420400","series":"1.2.826.0.1.3680043.8.498.48572788903787591848141713638081804289","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64022949342538714157641824612944253302","series":"1.2.826.0.1.3680043.8.498.36983293267458786187579569717074502992","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64066465191710322298540005681534170036","series":"1.2.826.0.1.3680043.8.498.37543665007390806194856756843810394413","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.55787414140753925274618680298630905599","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734","series":"1.2.826.0.1.3680043.8.498.10163718697149610968936392019419895558","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715","series":"1.2.826.0.1.3680043.8.498.13169754981090483540437046811517453047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477","series":"1.2.826.0.1.3680043.8.498.11512974439057827025206859411800435638","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676","series":"1.2.826.0.1.3680043.8.498.11386792283600257049020166426940918699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64487230511522940587342058274736571996","series":"1.2.826.0.1.3680043.8.498.25939401701241153369536462677744008867","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64527059976573114931020581328924993069","series":"1.2.826.0.1.3680043.8.498.10178052926196150995450702638798580832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223","series":"1.2.826.0.1.3680043.8.498.10523014616190443159239233747986703955","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188","series":"1.2.826.0.1.3680043.8.498.11810869323466957201411917434499364553","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64752580249608101443758975861213065048","series":"1.2.826.0.1.3680043.8.498.76823121465955116839506873082436080995","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64917073507426524387269500871970032783","series":"1.2.826.0.1.3680043.8.498.64044734792639838778477113202584950747","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183","series":"1.2.826.0.1.3680043.8.498.47691155533480590944043770858592256561","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65085954283273471512936799274064949557","series":"1.2.826.0.1.3680043.8.498.12541496840062320882484127068035697433","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65124952546314712352837396207287384405","series":"1.2.826.0.1.3680043.8.498.59678900321011366653765254083672756444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325","series":"1.2.826.0.1.3680043.8.498.11207657898480204641679447674534219992","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65244223225926362862970496735749098549","series":"1.2.826.0.1.3680043.8.498.82015422571326947421905515983932142932","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460","series":"1.2.826.0.1.3680043.8.498.11456365546888972724874062125587296514","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554","series":"1.2.826.0.1.3680043.8.498.90349736851508037752026531968055715860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65530757406333318449497070048719890442","series":"1.2.826.0.1.3680043.8.498.11346422492424270430816698579159315958","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856","series":"1.2.826.0.1.3680043.8.498.11195583465346944862811848334394040450","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65667678378249093118816281743956646856","series":"1.2.826.0.1.3680043.8.498.11215265872843715688969868622328007788","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327","series":"1.2.826.0.1.3680043.8.498.51563509636661462327570594061418684481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65763261177572072944828171262268039852","series":"1.2.826.0.1.3680043.8.498.51862577158388999698554214283119016496","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65788206282522750507559682036750692505","series":"1.2.826.0.1.3680043.8.498.11090102870867460571881314219419785698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65834774574423727070722226151408235786","series":"1.2.826.0.1.3680043.8.498.28146133306189783010052215178855557320","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65896962360688978581417748865609785541","series":"1.2.826.0.1.3680043.8.498.28697002973153351537054665711678875505","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65959182350599581687336889990076028593","series":"1.2.826.0.1.3680043.8.498.82418341380517912098084905365302434466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66022464665791424345622894674375661089","series":"1.2.826.0.1.3680043.8.498.52104402233043529791479285117035775594","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66065279867975557954375204776779208827","series":"1.2.826.0.1.3680043.8.498.56559430386428239403604339794002036242","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66156015559325316834458423557757995415","series":"1.2.826.0.1.3680043.8.498.12585120150326130872972573006441660828","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769","series":"1.2.826.0.1.3680043.8.498.16559909695971207596214701902671506088","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755","series":"1.2.826.0.1.3680043.8.498.90845626751521246412769710780428058666","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66308195780676793917989368423009849637","series":"1.2.826.0.1.3680043.8.498.89199423581099883939653881729242877802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66334686028272068369345540785320291695","series":"1.2.826.0.1.3680043.8.498.54690476002007218895568660117966336039","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66404725797392899950170138361835649917","series":"1.2.826.0.1.3680043.8.498.78608631124278554323894064976597707002","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66510565421813721258189101326477318907","series":"1.2.826.0.1.3680043.8.498.14524717636522150829765687348588250952","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66640124635222298792902508374800337726","series":"1.2.826.0.1.3680043.8.498.87667636113043989845428422332220686924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66737859593831436259089736126768464771","series":"1.2.826.0.1.3680043.8.498.75929621415909800232155632481242012920","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524","series":"1.2.826.0.1.3680043.8.498.12072541383862072442516070262333596334","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66973089869591928349547584554030589647","series":"1.2.826.0.1.3680043.8.498.12811174271090373168597952394892070029","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.75400056114511267591792237629525617008","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713","series":"1.2.826.0.1.3680043.8.498.10479787399521690017359241486980467950","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67189969532330535369638450306601659967","series":"1.2.826.0.1.3680043.8.498.11884598424174300852967306772358545434","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67281816624401881515444587844116136597","series":"1.2.826.0.1.3680043.8.498.66813179329778705592692785059725045027","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678","series":"1.2.826.0.1.3680043.8.498.11249370563297630608594408935440863615","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784","series":"1.2.826.0.1.3680043.8.498.62209322614278118294773741892554635999","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.13131404906732628102839591109680642861","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67527486841399075490744039058207332725","series":"1.2.826.0.1.3680043.8.498.93123116136195924368525966156087723657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67601368724828563564325678719126190797","series":"1.2.826.0.1.3680043.8.498.14792789514188114977884276314769720776","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.73926143085093235845407144850145500557","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67705068860320494485379949506807098172","series":"1.2.826.0.1.3680043.8.498.11441652382915702972273038283599000118","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67740000270419228186274186410889719154","series":"1.2.826.0.1.3680043.8.498.34861496853371447683281049010386510965","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67775301856843451215029610725114809378","series":"1.2.826.0.1.3680043.8.498.13119231995932072434637450001033725215","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228","series":"1.2.826.0.1.3680043.8.498.48434517533851752817474873896250552020","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67901792488494108577057563458320460313","series":"1.2.826.0.1.3680043.8.498.13311901221572052140857223559263741465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.10007033060853101402914165412426592663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67981157392064878495682078197161506255","series":"1.2.826.0.1.3680043.8.498.14758551219936823540470816811539809767","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669","series":"1.2.826.0.1.3680043.8.498.54668881829125877160713642750832516398","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.11504656323318458047204951956423360619","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68131564130881160284492534203572971623","series":"1.2.826.0.1.3680043.8.498.34618280583660839021123347870821919519","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.13666733115662526243643968035475425416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68176786698913278568384612123261229708","series":"1.2.826.0.1.3680043.8.498.84591804477869694078117336353548551631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68236154531625015457518732674625182872","series":"1.2.826.0.1.3680043.8.498.10185698269969452545400148059412363717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68264559730756309944466784364591509088","series":"1.2.826.0.1.3680043.8.498.12331290755174969003349791468024886686","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68328371080087298795966933114044350359","series":"1.2.826.0.1.3680043.8.498.92405892450464181435823282912945269838","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68434323590918408241182252996136001891","series":"1.2.826.0.1.3680043.8.498.63657633288365140528272585154813117066","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68442883836038453562548037358230921748","series":"1.2.826.0.1.3680043.8.498.71307200769544253955025994514135264089","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.12266552908429747682251232311026317189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028","series":"1.2.826.0.1.3680043.8.498.51074834415865498105921963551840335454","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68637419175382604890136839259344633105","series":"1.2.826.0.1.3680043.8.498.13195359019903028749524893891811095709","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68707023438918237611039069622720041636","series":"1.2.826.0.1.3680043.8.498.67694592979355125885274188847907706765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835","series":"1.2.826.0.1.3680043.8.498.24982461566242329645366127653765611677","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68806105227826942108149090508198910591","series":"1.2.826.0.1.3680043.8.498.87022660898953271607677050762644331869","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68919687595587004665658610968739852899","series":"1.2.826.0.1.3680043.8.498.32689130449795275823116907458871647185","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68969263295435881669720950631709959471","series":"1.2.826.0.1.3680043.8.498.11337729390209490819525006507805453550","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840","series":"1.2.826.0.1.3680043.8.498.74250432972767945533719602327377155888","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69131433601281694165384475765166382765","series":"1.2.826.0.1.3680043.8.498.12674313096538330554659529722601443307","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69212713525853754921729012449897924087","series":"1.2.826.0.1.3680043.8.498.11305867493551337986909659208456872525","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198","series":"1.2.826.0.1.3680043.8.498.11054866632685669561481448318288894390","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002","series":"1.2.826.0.1.3680043.8.498.29181677329774151528489913812825347708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69411539430142524612470544041263824359","series":"1.2.826.0.1.3680043.8.498.10641790822990412415774346494981677158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69470130842235635628917024269527211321","series":"1.2.826.0.1.3680043.8.498.91586168363309057011012255529469535740","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69544085715595825083308172629360899339","series":"1.2.826.0.1.3680043.8.498.11196343239270276807411581914910806616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69635429172929809539574727887787303282","series":"1.2.826.0.1.3680043.8.498.11463994058138489879444246526361933946","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69665129458102359604411189244405850661","series":"1.2.826.0.1.3680043.8.498.80471485491050189066622321437479256159","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813","series":"1.2.826.0.1.3680043.8.498.13302003063447406229374047585488123192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69828573582708185393942443598783108800","series":"1.2.826.0.1.3680043.8.498.10286189646672478142339160193945490916","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69917156795941398965793962392683899162","series":"1.2.826.0.1.3680043.8.498.93975515961464374478472290023222296880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063","series":"1.2.826.0.1.3680043.8.498.38156015932787767312483149426437347692","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70129413051088864999976563404818211137","series":"1.2.826.0.1.3680043.8.498.11212439766926866438626116423103089447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70271042033022764422915852922035168712","series":"1.2.826.0.1.3680043.8.498.82695599833196748291216726802786407751","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70310491987092072540988486213408328865","series":"1.2.826.0.1.3680043.8.498.94571781280254513538844529059251263903","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70375834273924102402706635540898552043","series":"1.2.826.0.1.3680043.8.498.97646367735711330982976230124941030314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70435722790459313233345196619727661231","series":"1.2.826.0.1.3680043.8.498.10140079655001801960882726230771828753","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894","series":"1.2.826.0.1.3680043.8.498.10662922911561040788259715257688056551","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70531311408047627903498280968772932092","series":"1.2.826.0.1.3680043.8.498.16106904151716621065359724993776973327","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827","series":"1.2.826.0.1.3680043.8.498.68821764706264194749138852971156901902","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717","series":"1.2.826.0.1.3680043.8.498.58333005650305151977822895037007164040","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70831735578941318362881379802693293952","series":"1.2.826.0.1.3680043.8.498.13175278840317707505173842156687669707","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541","series":"1.2.826.0.1.3680043.8.498.11861176148907478147591964017998776610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266","series":"1.2.826.0.1.3680043.8.498.68924390109439410258520854040298411935","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068","series":"1.2.826.0.1.3680043.8.498.22565264868327494191343521591345966467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70948162965271404678418299669321564765","series":"1.2.826.0.1.3680043.8.498.38422327977131686393128916683702234453","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70966951020892708303866968206211604316","series":"1.2.826.0.1.3680043.8.498.60764632888268489954617321326849737362","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71047916312801688293181464016099792199","series":"1.2.826.0.1.3680043.8.498.42897932098095457440809124781733018052","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71084400653875583977681616665093993652","series":"1.2.826.0.1.3680043.8.498.39487532583414016171889993977221409527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71262022207903748073532915013888861999","series":"1.2.826.0.1.3680043.8.498.88849630443684052374304969697232134820","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71388532549899713881913113571627425347","series":"1.2.826.0.1.3680043.8.498.17946477973863172161784136282217216684","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237","series":"1.2.826.0.1.3680043.8.498.14093023822572174395820902562537358053","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629","series":"1.2.826.0.1.3680043.8.498.44737592293067613729981006184946160660","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71671013221669628598634284971059228267","series":"1.2.826.0.1.3680043.8.498.26758264620565880283048849434064009554","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610","series":"1.2.826.0.1.3680043.8.498.99206855316961800031103786874279710359","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71709477515434076105531625950744135656","series":"1.2.826.0.1.3680043.8.498.37919564258066401435626724982350214278","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366","series":"1.2.826.0.1.3680043.8.498.18508292456157718837942998439367692675","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71766161038805688022754893134126528749","series":"1.2.826.0.1.3680043.8.498.84485606560011247098859366407509781685","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914","series":"1.2.826.0.1.3680043.8.498.21181460900910515511564203597958886590","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.15776442834014620829960104396949165032","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71901039762584748047172479049347609017","series":"1.2.826.0.1.3680043.8.498.28957047480324246658684516078792670603","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72008869367139958325145888733651022033","series":"1.2.826.0.1.3680043.8.498.12840977446951528128092112172580565907","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72018602547183896629903902899903468287","series":"1.2.826.0.1.3680043.8.498.47938211459751283209526971914452766492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597","series":"1.2.826.0.1.3680043.8.498.37216112168491137221108352607345413303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641","series":"1.2.826.0.1.3680043.8.498.99903185831825418113284047949375040183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72237065473467452226073540784070938342","series":"1.2.826.0.1.3680043.8.498.37097810089163389968193338232438478758","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401","series":"1.2.826.0.1.3680043.8.498.11405695859142582680528058569598039489","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72276631472244076773626607623783737687","series":"1.2.826.0.1.3680043.8.498.10483567027444360172241675960613984189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72368645606329000923061796699431437924","series":"1.2.826.0.1.3680043.8.498.88223425776225907293112157213900378430","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72395496925770440630088720587484704971","series":"1.2.826.0.1.3680043.8.498.57023572950068060909730125543988922989","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72494404099796908482534991590660770670","series":"1.2.826.0.1.3680043.8.498.36762889568125976260024942400309924181","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203","series":"1.2.826.0.1.3680043.8.498.51827546610914149387144722414971286141","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72587440202779699193620339987034971895","series":"1.2.826.0.1.3680043.8.498.95414606418229087353190403478173011139","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72680485820462987585370556994612687540","series":"1.2.826.0.1.3680043.8.498.63538072703095663567179407892616848610","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639","series":"1.2.826.0.1.3680043.8.498.29496807740583474174127848544895682830","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72786773761211751423503785260617678955","series":"1.2.826.0.1.3680043.8.498.10828187660323389296140564899602708308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921","series":"1.2.826.0.1.3680043.8.498.45433966192867460741417444204369175868","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72896348955280307080916504120667484813","series":"1.2.826.0.1.3680043.8.498.10183002382366122196721283076964117231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72937134140407212536042212109839895294","series":"1.2.826.0.1.3680043.8.498.38110659891488547488964936377639840909","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331","series":"1.2.826.0.1.3680043.8.498.43839962888177349190898371887133964640","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.24168888324252380403473219324103275057","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.40112320787298868336827193546891114149","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73101997284755549124715401557235981642","series":"1.2.826.0.1.3680043.8.498.76955242876062827500349963881210494833","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73198335328613992289471885012751338458","series":"1.2.826.0.1.3680043.8.498.81672351643649842035711867303496309435","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73215265387578908341609746559916587894","series":"1.2.826.0.1.3680043.8.498.14303515791053304768629826309207836571","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73242978867957397254276973409271434573","series":"1.2.826.0.1.3680043.8.498.33090754016238252755216205674483434685","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930","series":"1.2.826.0.1.3680043.8.498.33077713529712590859236571991193524951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73325959289998105909842627876607163467","series":"1.2.826.0.1.3680043.8.498.70437630803180705559047786560336463897","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284","series":"1.2.826.0.1.3680043.8.498.11535146618236209036136293933646945054","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803","series":"1.2.826.0.1.3680043.8.498.80236168490086489978039862234395645008","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73522298807198784022883320957252530465","series":"1.2.826.0.1.3680043.8.498.33843891698739999189974505625445801521","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73545582819102316606081606511277579853","series":"1.2.826.0.1.3680043.8.498.17791845061698739052076736629151053407","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73602922823389605007702521335136347605","series":"1.2.826.0.1.3680043.8.498.93247798304758718686949250787271765100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73716752081143767997507440278231972668","series":"1.2.826.0.1.3680043.8.498.26414842479650576012151824934081252982","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478","series":"1.2.826.0.1.3680043.8.498.81196521174806912309575752314615178439","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73891117356545433634964184054304516063","series":"1.2.826.0.1.3680043.8.498.82046558525392297116647101018253711295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73988798714023285209290281409329527637","series":"1.2.826.0.1.3680043.8.498.10648484734781068429322813243070900614","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74078340346189210015785395961373387782","series":"1.2.826.0.1.3680043.8.498.37193319081973432642865084706314952004","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520","series":"1.2.826.0.1.3680043.8.498.89522721497073176034742384681822112012","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74177736542522869266952610634091628727","series":"1.2.826.0.1.3680043.8.498.22773066600979694876546102816633562352","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871","series":"1.2.826.0.1.3680043.8.498.18354754151030512388464953923092249898","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604","series":"1.2.826.0.1.3680043.8.498.11944366496082019521893786416350376696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520","series":"1.2.826.0.1.3680043.8.498.47211775256507609420462639906237348825","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74282280145383357786409764912398881272","series":"1.2.826.0.1.3680043.8.498.50037238540850686361623309290986793682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860","series":"1.2.826.0.1.3680043.8.498.19736922186838321245822290713631090027","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74423888576992387013857540016155410962","series":"1.2.826.0.1.3680043.8.498.39801519438179280221162298675138638850","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74471396469132211069109289423495141062","series":"1.2.826.0.1.3680043.8.498.79177423805534441113379435017475507527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74622912009754642425258484521795653023","series":"1.2.826.0.1.3680043.8.498.10239051814937800544006519348365595661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74714018385278253151175853998945654373","series":"1.2.826.0.1.3680043.8.498.53211051985426633949463280825085920960","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74855168460956358728708865599401983005","series":"1.2.826.0.1.3680043.8.498.71616242137943157017463551126010501082","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74872419726042767543822107291861307866","series":"1.2.826.0.1.3680043.8.498.77579609804677320689204666751079543657","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312","series":"1.2.826.0.1.3680043.8.498.30900354749422264395375231810870199828","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75067227246352035607675978826177487284","series":"1.2.826.0.1.3680043.8.498.10564280018678910314584588910195940596","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75186274888329123179129240384960518861","series":"1.2.826.0.1.3680043.8.498.44452671242744052346648356900093488755","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309","series":"1.2.826.0.1.3680043.8.498.89908887188589585530809196305518156663","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681","series":"1.2.826.0.1.3680043.8.498.84612196935666939723686188644280933220","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75418637314688036152637525301534342254","series":"1.2.826.0.1.3680043.8.498.15615882028979402205308200550365677278","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75427432788337288702609576920198810880","series":"1.2.826.0.1.3680043.8.498.54812901167403001567998811681194049142","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75522002047190283617659083787085821064","series":"1.2.826.0.1.3680043.8.498.10465858513846012282125875866089375386","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75548650589349966221875964542604002971","series":"1.2.826.0.1.3680043.8.498.15615248797948991816547968905934821388","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.10311356464066352598672427731045737175","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.13017815467576660104515442764701800427","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75739561647115166735277767560308263446","series":"1.2.826.0.1.3680043.8.498.12284508570978487922310052861550323692","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75746520601616488682662195284941583868","series":"1.2.826.0.1.3680043.8.498.12147085360584798190722558646980268399","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864","series":"1.2.826.0.1.3680043.8.498.48470319361772558989419708775140105708","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877","series":"1.2.826.0.1.3680043.8.498.11032972829430446606848197086920630621","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75976388060969166980873144500539735332","series":"1.2.826.0.1.3680043.8.498.45455661467955482020008366356956417800","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386","series":"1.2.826.0.1.3680043.8.498.11886651523855996282772219534325930202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523","series":"1.2.826.0.1.3680043.8.498.97693374381584989555811399769901257533","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.72658887322131335959033608678017568505","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76190704217737835083438638928129775625","series":"1.2.826.0.1.3680043.8.498.99124093166091580590596295749343107932","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76281930374588438789344170829280438697","series":"1.2.826.0.1.3680043.8.498.92329367835011294880082228019333879527","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76350662767360412185091180512006419379","series":"1.2.826.0.1.3680043.8.498.27796575097549552015009851265951807839","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76525107909962775502207829870873494024","series":"1.2.826.0.1.3680043.8.498.69499841173831830722299576243390242958","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992","series":"1.2.826.0.1.3680043.8.498.22616104310897234807541023952167164077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76662146598924492573212477881527658730","series":"1.2.826.0.1.3680043.8.498.42562307096807480440582780767900051710","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76742750470853901575934097885382242952","series":"1.2.826.0.1.3680043.8.498.70699768361097883333927441885658473564","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909","series":"1.2.826.0.1.3680043.8.498.10048391752934964148396885796286084490","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737","series":"1.2.826.0.1.3680043.8.498.10194576796720221017523446481088800668","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254","series":"1.2.826.0.1.3680043.8.498.16356813141564841725754198246705832318","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77018355356797290743788712046815540223","series":"1.2.826.0.1.3680043.8.498.12499147485475642526940449655259417934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77058587552329832558354544874120592826","series":"1.2.826.0.1.3680043.8.498.11514515052342958225596236032328818886","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695","series":"1.2.826.0.1.3680043.8.498.58221254506928171421004312796174829621","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733","series":"1.2.826.0.1.3680043.8.498.13149883948772538380701678222279639612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.66598205083440700990920677492233300702","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044","series":"1.2.826.0.1.3680043.8.498.13167825534703979788720895941750726542","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77283634197289382453511269950488893300","series":"1.2.826.0.1.3680043.8.498.13311320340813069761859693034234690809","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485","series":"1.2.826.0.1.3680043.8.498.11535866189105373187953251583112414231","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.32323247418092317262462994831418091573","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012","series":"1.2.826.0.1.3680043.8.498.12509695703732158342282005635643350237","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512","series":"1.2.826.0.1.3680043.8.498.87297928186494716638082820305632194467","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77442647074650043391818472068835112806","series":"1.2.826.0.1.3680043.8.498.58225622983802500816642313850197138954","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77508762938574470575737016745357126587","series":"1.2.826.0.1.3680043.8.498.39007452210258542976719231498224710403","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77524507552350448792970399203131998592","series":"1.2.826.0.1.3680043.8.498.10870027068023756688896462425861470588","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727","series":"1.2.826.0.1.3680043.8.498.51526529666030495342692054713961228946","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77593859518024366170735307975173961777","series":"1.2.826.0.1.3680043.8.498.77012535005888356437795639079177188091","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176","series":"1.2.826.0.1.3680043.8.498.55252514720252387777203980363242721334","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77735690782343511872544703464260892488","series":"1.2.826.0.1.3680043.8.498.32944815998225255003854545848179981008","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77753753872601674668756777373761660873","series":"1.2.826.0.1.3680043.8.498.11256005678466724080630564734780676931","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387","series":"1.2.826.0.1.3680043.8.498.91163857143903322631244904620997588021","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77843765997593346714211135369376676497","series":"1.2.826.0.1.3680043.8.498.14537597152983416098187219169872488917","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77911973519830915742655727368286737079","series":"1.2.826.0.1.3680043.8.498.11187077266697114608942014214333223204","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78003548255041391467292087330671945247","series":"1.2.826.0.1.3680043.8.498.59588556763724059405597990150667394567","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78072765790191594685273447833272737429","series":"1.2.826.0.1.3680043.8.498.48390665005258578548353960308420960860","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78084594560404633421210837175859618437","series":"1.2.826.0.1.3680043.8.498.44673210757772776101594814355444588537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411","series":"1.2.826.0.1.3680043.8.498.11247153284812327269043986976308353516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78238394512811614973689011297363399053","series":"1.2.826.0.1.3680043.8.498.92435677994266007731053253297642124651","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78286395605671977872377352305890466464","series":"1.2.826.0.1.3680043.8.498.13198575563966973092331424712506571016","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78420763928408680447797586163977368190","series":"1.2.826.0.1.3680043.8.498.34155853566233321122789194875024803059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568","series":"1.2.826.0.1.3680043.8.498.13101524764059595716328033792804524360","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78521396404736679127326025240279747196","series":"1.2.826.0.1.3680043.8.498.12041425747633242534661825969925803771","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756","series":"1.2.826.0.1.3680043.8.498.18243432610279717161105666205572583562","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78674598286586525734293563394409166190","series":"1.2.826.0.1.3680043.8.498.97231708962369715276414468850402358303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052","series":"1.2.826.0.1.3680043.8.498.10146398894693788225839208270296720140","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78808320605816811006382053060951793555","series":"1.2.826.0.1.3680043.8.498.12212874616452345928819640525320387131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949","series":"1.2.826.0.1.3680043.8.498.35583271336362198185540889532659142888","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887","series":"1.2.826.0.1.3680043.8.498.43461609722438720569221001024103872658","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79080521163097116147379189510104170611","series":"1.2.826.0.1.3680043.8.498.10338028626209127232338182737552399496","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79122189963582241618519333445055548347","series":"1.2.826.0.1.3680043.8.498.33296195806639464229888836894417603664","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79224785668205420138951103045759609475","series":"1.2.826.0.1.3680043.8.498.10186260679271536022870228965898116480","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718","series":"1.2.826.0.1.3680043.8.498.15906064926552609457262547118401888435","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79258497507223824227830886693749460348","series":"1.2.826.0.1.3680043.8.498.12534224403806000543223446955668974892","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.25755074934969872546200523029047101535","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79326627674334637493320764205397279674","series":"1.2.826.0.1.3680043.8.498.12727775635901093202102368153501046089","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347","series":"1.2.826.0.1.3680043.8.498.67138271979268743170884708395545124769","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803","series":"1.2.826.0.1.3680043.8.498.25938194685996599023816009706232632024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807","series":"1.2.826.0.1.3680043.8.498.11384695274981718302093806564344517135","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79648517065511897282938523065537768426","series":"1.2.826.0.1.3680043.8.498.10399745662140806283692964597613951823","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775","series":"1.2.826.0.1.3680043.8.498.87565201875385962233584381961959494496","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79829033430619597320769579608450393290","series":"1.2.826.0.1.3680043.8.498.12719682076531989948068038041779850577","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668","series":"1.2.826.0.1.3680043.8.498.60823145998597667335588239356532504608","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80146250663015309456767479960653638934","series":"1.2.826.0.1.3680043.8.498.10251284258779540589661026563101041123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80180578697901833764761405542790185180","series":"1.2.826.0.1.3680043.8.498.36479567505871682810355401482029719420","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721","series":"1.2.826.0.1.3680043.8.498.10875915855834938567350578944508347249","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798","series":"1.2.826.0.1.3680043.8.498.12406947255418396228215356174649417643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111","series":"1.2.826.0.1.3680043.8.498.72090217567502176302926821111639400682","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774","series":"1.2.826.0.1.3680043.8.498.12378788387362440197265178959038549579","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80433141990021472112645841723991584182","series":"1.2.826.0.1.3680043.8.498.26243818707626922878834628157931142181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80492324225032367918842505447056622561","series":"1.2.826.0.1.3680043.8.498.77164658011963726158152638240786401739","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80557835452734362083552169469427388003","series":"1.2.826.0.1.3680043.8.498.12679140843771621047127466728471125277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80612964209397056653334784776523875647","series":"1.2.826.0.1.3680043.8.498.85461031908976979603973131354970495822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80642348849062865125509773837278900750","series":"1.2.826.0.1.3680043.8.498.78232388885212703245058727912067824595","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506","series":"1.2.826.0.1.3680043.8.498.89497915157636217099633086102027796519","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80698510014594772882040509716764819716","series":"1.2.826.0.1.3680043.8.498.31553326482145637113764783420659851047","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.80741358365199721720335324482078866565","series":"1.2.826.0.1.3680043.8.498.82574082899176232958924054638060671517","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80851365368114686105558347899990395309","series":"1.2.826.0.1.3680043.8.498.32931358222823219750590868653691849666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80876235930771882271600826613593202364","series":"1.2.826.0.1.3680043.8.498.33798630489622841077203441105432437352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417","series":"1.2.826.0.1.3680043.8.498.38433540242237740463466380052227936393","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057","series":"1.2.826.0.1.3680043.8.498.10285700314227499870878223073238087144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81096597169632636454986988944050298701","series":"1.2.826.0.1.3680043.8.498.10246148683881713170453801075572926087","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924","series":"1.2.826.0.1.3680043.8.498.11091564458068807508469889888483926250","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859","series":"1.2.826.0.1.3680043.8.498.69110538132396334346412987475904010312","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.14736386539065345605837224847895042662","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694","series":"1.2.826.0.1.3680043.8.498.17312914648574549262946250451161487954","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81464322239381998685473520085495957135","series":"1.2.826.0.1.3680043.8.498.10730741850384351684321667236647791777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81475058647553697318837239968139077272","series":"1.2.826.0.1.3680043.8.498.18403190251768213710564784079995137700","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81567030584250750745410094870790052181","series":"1.2.826.0.1.3680043.8.498.62712776720194793319652220456700775753","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.11803382369827350266456786973715571246","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81651455914141365028015556083080638757","series":"1.2.826.0.1.3680043.8.498.13351003287754033380273385838479538911","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81698411330214788362326470979160431674","series":"1.2.826.0.1.3680043.8.498.32964817889708485987971383272337524186","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81740348974208459068937862799489495782","series":"1.2.826.0.1.3680043.8.498.11065631349837233439681144705455411467","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81796385087779065457668769443782458844","series":"1.2.826.0.1.3680043.8.498.98368054701669452749315927089568125756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81819209302731920404237374320026807282","series":"1.2.826.0.1.3680043.8.498.30994061234555977690110131716979749201","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167","series":"1.2.826.0.1.3680043.8.498.16326990541306438901360997579028335406","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81957447654112577010261210572986132211","series":"1.2.826.0.1.3680043.8.498.12563600350439437585185471305734998251","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277","series":"1.2.826.0.1.3680043.8.498.13683882263483402923431768639824060927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678","series":"1.2.826.0.1.3680043.8.498.95163967175857022513221785372878527737","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82074883987134049971242727164506046476","series":"1.2.826.0.1.3680043.8.498.57462873381081977617223992402159921999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974","series":"1.2.826.0.1.3680043.8.498.11651771796636532465779430962418611996","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82169173362972793731001247413129649263","series":"1.2.826.0.1.3680043.8.498.10667331983065903522918176166210224430","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82355456395534835058857499033494900716","series":"1.2.826.0.1.3680043.8.498.36526472783457824113553564979287229867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82447100001417065190678617013741085481","series":"1.2.826.0.1.3680043.8.498.68282172753590498435441713825107556084","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051","series":"1.2.826.0.1.3680043.8.498.54958456184123846278366102765633884321","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82548952153746644951736669119607083230","series":"1.2.826.0.1.3680043.8.498.19132992504842639903904619846141323411","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82587763533601760002619014353812074700","series":"1.2.826.0.1.3680043.8.498.10000638804426494408367014941970083813","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914","series":"1.2.826.0.1.3680043.8.498.27507871846929471868839749746111558253","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758","series":"1.2.826.0.1.3680043.8.498.12114242047632352161110349831180358647","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925","series":"1.2.826.0.1.3680043.8.498.42898878199023886639210724339150230245","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83000639623679673469640458882929412717","series":"1.2.826.0.1.3680043.8.498.12538064500925848830181754494496429314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966","series":"1.2.826.0.1.3680043.8.498.79914667095927928930634729144892728014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83061811791652434833797599835671351097","series":"1.2.826.0.1.3680043.8.498.11171616656613236252804074968774214557","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83121378128521985713113311344685112825","series":"1.2.826.0.1.3680043.8.498.21503298417260679629197694085962818403","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332","series":"1.2.826.0.1.3680043.8.498.12738121495965167315326205589051818312","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.12873356086028632445667675260169597230","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83384018761005700404957421316476952190","series":"1.2.826.0.1.3680043.8.498.12093166304699961202779582720050455552","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936","series":"1.2.826.0.1.3680043.8.498.14189489827170224455552289119462436334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462","series":"1.2.826.0.1.3680043.8.498.20841604508834566928700995786980434997","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.11431101652481224918309316035151741859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83847285247209768917850605454140194088","series":"1.2.826.0.1.3680043.8.498.73851154771016623053715171877697429236","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84007436672323587547274248257117331532","series":"1.2.826.0.1.3680043.8.498.49399931156350465212168524202838649156","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783","series":"1.2.826.0.1.3680043.8.498.17167873424015958502570965392350793105","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399","series":"1.2.826.0.1.3680043.8.498.59558615374719347374124519503090476925","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.49757233778194150033043840758080244617","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84292569246911697141902796190329671261","series":"1.2.826.0.1.3680043.8.498.26434948058778766901134296005379149157","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84477198486144701928775090913243250655","series":"1.2.826.0.1.3680043.8.498.34886522784727035938372639401770502523","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475","series":"1.2.826.0.1.3680043.8.498.12270576772608715155402763792604932739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","series":"1.2.826.0.1.3680043.8.498.12865530870458558427176708022691982887","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84709539128629951937294117586370340042","series":"1.2.826.0.1.3680043.8.498.11427227625845558493629265265741506442","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132","series":"1.2.826.0.1.3680043.8.498.84559874088196087684358879407089822186","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84857368518703570476541203284655412396","series":"1.2.826.0.1.3680043.8.498.80038275108643538318607063383873711058","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84982310626395569925821771375535259394","series":"1.2.826.0.1.3680043.8.498.10633639238632449885042620581338265097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798","series":"1.2.826.0.1.3680043.8.498.11801899619330744962451494068793678881","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.17095798182826616633629450313143427647","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85194002080793317087210934629814349622","series":"1.2.826.0.1.3680043.8.498.15933268726194960283072482223202587078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047","series":"1.2.826.0.1.3680043.8.498.59206843762375351913484521474902011807","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85341019006931436802294963713199308935","series":"1.2.826.0.1.3680043.8.498.90323184314042386603002697590647890733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85397267100597758920280635204297906286","series":"1.2.826.0.1.3680043.8.498.11798873175975375987590258093462801856","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892","series":"1.2.826.0.1.3680043.8.498.90141615921223620640426184829064758844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85475892602877115812059264719695919613","series":"1.2.826.0.1.3680043.8.498.38908671446541057316299213360184527894","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85545102350084131396266710667808696561","series":"1.2.826.0.1.3680043.8.498.59787211561887265947777331819963593246","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762","series":"1.2.826.0.1.3680043.8.498.25635857167358856349236592575169354772","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85649172620970194090967653766415467836","series":"1.2.826.0.1.3680043.8.498.35679460518494535168091409217621778213","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.10261186076217295470513869215481298395","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85728170022176096748897349447671687164","series":"1.2.826.0.1.3680043.8.498.97839955907655673388512876762996145232","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85748882309364598407865143890978767674","series":"1.2.826.0.1.3680043.8.498.54744613804795343221499848700999797573","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800","series":"1.2.826.0.1.3680043.8.498.12144234657715518535270998341407692374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85882078780717905717989373451578844863","series":"1.2.826.0.1.3680043.8.498.13097195490216254682672887125286340457","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85932495800565018166769303205378781707","series":"1.2.826.0.1.3680043.8.498.79105280470784698172035613219109820756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85970222049966384156120176704985355996","series":"1.2.826.0.1.3680043.8.498.12870940226966535909071161426819652481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85992651772359515416050249463715355943","series":"1.2.826.0.1.3680043.8.498.75760296977068374515179218759799440552","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827","series":"1.2.826.0.1.3680043.8.498.59992515998173487364479424415989744099","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86082522186055358984181640226529351801","series":"1.2.826.0.1.3680043.8.498.87011089675464794216505422053877855715","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86216155858324761845740646421500394797","series":"1.2.826.0.1.3680043.8.498.12157394310280488867936428126466037712","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86251314013737658351649446042801810374","series":"1.2.826.0.1.3680043.8.498.27947262440678262377796455558231902580","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86280033511814602594654628677479528508","series":"1.2.826.0.1.3680043.8.498.49298430323078132680830534831944743338","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373","series":"1.2.826.0.1.3680043.8.498.22057108738287870257118757735923991434","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655","series":"1.2.826.0.1.3680043.8.498.10174995779448892080168914106259191543","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391","series":"1.2.826.0.1.3680043.8.498.32800735087079563814684465746394538725","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86435271359723045834079326733478538148","series":"1.2.826.0.1.3680043.8.498.87353984491120805802352812838209524424","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543","series":"1.2.826.0.1.3680043.8.498.14710728907323867090275923452758010165","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","series":"1.2.826.0.1.3680043.8.498.73178781644221443709008983379870378618","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86651345701812229172985086531423369421","series":"1.2.826.0.1.3680043.8.498.36188366441272388314154367740839790973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.11067041584623746886795717306570431803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037","series":"1.2.826.0.1.3680043.8.498.10638840135691883629383334909627986433","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930","series":"1.2.826.0.1.3680043.8.498.11586762459550408282642312338187559471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87064372531225943854092598173945043861","series":"1.2.826.0.1.3680043.8.498.50383782510102936025395657048635862759","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520","series":"1.2.826.0.1.3680043.8.498.10269066972536277128227334433333853820","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87208714421237491643915928765740747633","series":"1.2.826.0.1.3680043.8.498.12320080613325470076700518907564208067","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87222653734797522858506220026427900177","series":"1.2.826.0.1.3680043.8.498.84458604955858330872401067031901226569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.32222800028587006108998940362906029943","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87338030934656808032044071206920470285","series":"1.2.826.0.1.3680043.8.498.11682024941671187723901259611093498703","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87413924318793242918916384453755054267","series":"1.2.826.0.1.3680043.8.498.13287198277707927571697131553705092180","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155","series":"1.2.826.0.1.3680043.8.498.10312812987713129926690854187060355401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87550037046735743504446944390018580426","series":"1.2.826.0.1.3680043.8.498.25053923727275507939976809152732530821","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.70356018899723732846727639505213848652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015","series":"1.2.826.0.1.3680043.8.498.86016158831029702267842556545378714716","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87742737485142445536023876862256284839","series":"1.2.826.0.1.3680043.8.498.56384525772453691018056570789411941536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87769640167791104899516835133527936885","series":"1.2.826.0.1.3680043.8.498.81743164005311638273377586096457122103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87845297884497964690686366000533570840","series":"1.2.826.0.1.3680043.8.498.85124005190380580386149681296048921300","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87900330305833722536121052961334699743","series":"1.2.826.0.1.3680043.8.498.13350716120719232722603842586700733195","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88064086448060280325724227239585797761","series":"1.2.826.0.1.3680043.8.498.10742528169905751054836298786571514087","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905","series":"1.2.826.0.1.3680043.8.498.13396760576178235516752504610287964034","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689","series":"1.2.826.0.1.3680043.8.498.12876622998746350351648420775660844939","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894","series":"1.2.826.0.1.3680043.8.498.65034492375941651932597888589165197738","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88457076333783710706782784856816112502","series":"1.2.826.0.1.3680043.8.498.12857195070416255446991333776738514324","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.88480827968231439729532202780922145132","series":"1.2.826.0.1.3680043.8.498.10715714205607061691813142695245271573","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88565389747258713872180859306907392080","series":"1.2.826.0.1.3680043.8.498.10402131223843888288334718856087780792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88574487918084130981335189023288704909","series":"1.2.826.0.1.3680043.8.498.87251170387072660863904855213078981537","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88620225126852321191690360547272730103","series":"1.2.826.0.1.3680043.8.498.41243152634568655717958886348561859696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88655485247489827200671636011965281184","series":"1.2.826.0.1.3680043.8.498.25614776094994526107957643987068693469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88687994744258163246685527273559741656","series":"1.2.826.0.1.3680043.8.498.11660321067680648808340741845165140145","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88726804476694411363564819498296119763","series":"1.2.826.0.1.3680043.8.498.76834786879241380268407390382148294946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212","series":"1.2.826.0.1.3680043.8.498.90359010382771756838082309887228627657","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328","series":"1.2.826.0.1.3680043.8.498.45441486034819608613762447172474961876","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177","series":"1.2.826.0.1.3680043.8.498.87212900521141090055748513600749291559","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88962401400598535739494671683916272476","series":"1.2.826.0.1.3680043.8.498.84521845866280322078430696320079152854","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.10101040229452848020149459969845526907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294","series":"1.2.826.0.1.3680043.8.498.49666607384593474703368339048750030239","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116","series":"1.2.826.0.1.3680043.8.498.11969771238911427913240191179215458261","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89204062904097086723738217311755833877","series":"1.2.826.0.1.3680043.8.498.50876750268274095980916073996376665675","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428","series":"1.2.826.0.1.3680043.8.498.11332974718758043875906453906546624261","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530","series":"1.2.826.0.1.3680043.8.498.41012501818143710788582191761619967997","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89350648596674998203116057439419132072","series":"1.2.826.0.1.3680043.8.498.34330895619976945089813847714004642866","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89416235692252764852783739140428958560","series":"1.2.826.0.1.3680043.8.498.11418544320352995631696830390308301312","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89445342796957655182687412965056270903","series":"1.2.826.0.1.3680043.8.498.52825105371101731934652615087487003927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334","series":"1.2.826.0.1.3680043.8.498.45674211335570666859281187855782565886","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646","series":"1.2.826.0.1.3680043.8.498.13843187397354454893893732521214831228","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89746004948304570186940283501211828475","series":"1.2.826.0.1.3680043.8.498.62213597284882442511926162531351512759","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89842390998700046088751402253903762827","series":"1.2.826.0.1.3680043.8.498.63434560813332624492472815004173345615","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89935384074170571164295050203650829722","series":"1.2.826.0.1.3680043.8.498.12691221776421169935886600136487252813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89967772422936726289461201791871771011","series":"1.2.826.0.1.3680043.8.498.65994939942196703776845322053746029468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032","series":"1.2.826.0.1.3680043.8.498.72769757611972915966372483766141318132","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90103210546658057701715546323552886913","series":"1.2.826.0.1.3680043.8.498.92560135311999908035708885796545321172","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947","series":"1.2.826.0.1.3680043.8.498.42852826856265423562639323734012327741","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90206716031776606956011363950625953310","series":"1.2.826.0.1.3680043.8.498.11141470347110722444321778645819095406","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90251533154603802045666102095577144628","series":"1.2.826.0.1.3680043.8.498.10575756243848744418205761619783778142","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90268709354762000249148795667759483876","series":"1.2.826.0.1.3680043.8.498.49837816785490281283378595689461665488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90288598546566291188788375261370919222","series":"1.2.826.0.1.3680043.8.498.77464044859524343070000923488468715621","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942","series":"1.2.826.0.1.3680043.8.498.18141808203569639170489395213024835997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90342120331229511020327052213821820086","series":"1.2.826.0.1.3680043.8.498.68219895246539134987475998488519472605","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010","series":"1.2.826.0.1.3680043.8.498.82761853421080972615640906040307526894","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90490891782778285565462083536468192410","series":"1.2.826.0.1.3680043.8.498.51238116107389153779560110640327706172","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.16200009883628005320849442486548972003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080","series":"1.2.826.0.1.3680043.8.498.11967494220047402705594437482534257036","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90679199017361780574384378857945322240","series":"1.2.826.0.1.3680043.8.498.64459513864959425666569967766318231441","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839","series":"1.2.826.0.1.3680043.8.498.97171532594035373898534676709283631659","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979","series":"1.2.826.0.1.3680043.8.498.11526359164738237174851246811265488942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296","series":"1.2.826.0.1.3680043.8.498.78624419079864366994276244942836863953","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91106275065657355304239097295699332135","series":"1.2.826.0.1.3680043.8.498.13357281761048888090570681200821823993","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91124979914013108948829704350601987380","series":"1.2.826.0.1.3680043.8.498.44722362452268813843091908111549216852","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454","series":"1.2.826.0.1.3680043.8.498.34449190612240471830130429580667013085","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91206973544724011503639050238118945528","series":"1.2.826.0.1.3680043.8.498.98349989687034340694145529838922342775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91237082716839283641165609666644466929","series":"1.2.826.0.1.3680043.8.498.93252310037884334842588584982995537933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089","series":"1.2.826.0.1.3680043.8.498.58726687678306824313467263953174717206","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365","series":"1.2.826.0.1.3680043.8.498.42187776906573834309485068980364808328","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042","series":"1.2.826.0.1.3680043.8.498.12851763333340747284867732483958585542","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.95581893445927535243292420591118251737","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579","series":"1.2.826.0.1.3680043.8.498.23785450805727872389118576071634314819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91611403539216247154562810076063878924","series":"1.2.826.0.1.3680043.8.498.87690798358011185006723263143452919833","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037","series":"1.2.826.0.1.3680043.8.498.10665746250129896011687890318657807820","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91744384920314309659802891351055961910","series":"1.2.826.0.1.3680043.8.498.92612917056879583698764297098091109186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91883829404782805058767342767479130421","series":"1.2.826.0.1.3680043.8.498.22651814983484020300565489845093776524","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91910585599088493175245713843727208369","series":"1.2.826.0.1.3680043.8.498.86977750249422633435884400426298809611","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.11018061973861079072736753223379511583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069","series":"1.2.826.0.1.3680043.8.498.95606761382746416238097454783170097184","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92126358259118823397932994812520542494","series":"1.2.826.0.1.3680043.8.498.10825908297967828427203546164069411487","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675","series":"1.2.826.0.1.3680043.8.498.73758050659621838380657253423368695892","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164","series":"1.2.826.0.1.3680043.8.498.50083368074771359569133756129569196619","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728","series":"1.2.826.0.1.3680043.8.498.76919350567825536633087752498947641003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.58363623852593675309720163219829122967","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92455257597708674644536154732642099964","series":"1.2.826.0.1.3680043.8.498.45921943275767856718390301150982272409","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92602515666185999130675702469977883616","series":"1.2.826.0.1.3680043.8.498.86257032318592333503686098177891694259","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095","series":"1.2.826.0.1.3680043.8.498.74125508440977419669802760068672778900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780","series":"1.2.826.0.1.3680043.8.498.92317624653422077942782399883199333424","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92722393862174380736027353110424417588","series":"1.2.826.0.1.3680043.8.498.31696196908278178946151455093414679833","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078","series":"1.2.826.0.1.3680043.8.498.27283781244079080899021348391672473328","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.70649239796527400722399907758429026565","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93062749017302143105207135454940053142","series":"1.2.826.0.1.3680043.8.498.11689082644240368070207016855981106795","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93104685852024334015651558514195715956","series":"1.2.826.0.1.3680043.8.498.23018397931728158376889231807046167421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677","series":"1.2.826.0.1.3680043.8.498.79699901089415742413404456747117749427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93222021909191248270818335069738080963","series":"1.2.826.0.1.3680043.8.498.76419718114353661735706755958302065971","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93318293884671598055223778443354732793","series":"1.2.826.0.1.3680043.8.498.11163740575356910789945404908097252738","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93359387652821116106937882388838959209","series":"1.2.826.0.1.3680043.8.498.59940349175224102823331352345714682235","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679","series":"1.2.826.0.1.3680043.8.498.15110105565875395767324671061324093089","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233","series":"1.2.826.0.1.3680043.8.498.77775840622022368667611494040439989497","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93719148934772792523193842156217224487","series":"1.2.826.0.1.3680043.8.498.44520431759435262325997607233504096478","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93728117237268000947508816107096388086","series":"1.2.826.0.1.3680043.8.498.10153784311462065548870801663076432323","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418","series":"1.2.826.0.1.3680043.8.498.22377476551065187183070072370363779409","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308","series":"1.2.826.0.1.3680043.8.498.80221756082753170009711470572576342545","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94002633501154835018549300667479350401","series":"1.2.826.0.1.3680043.8.498.45507152696800888064484232200683598450","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","series":"1.2.826.0.1.3680043.8.498.11897092676257644574033344955052093768","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94244885518758327449236878653096161475","series":"1.2.826.0.1.3680043.8.498.28663441371657689769701265207802685605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94274898137310893945218731717226819541","series":"1.2.826.0.1.3680043.8.498.11832458387130737461987905889931073027","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94328216642444410827846344286227627425","series":"1.2.826.0.1.3680043.8.498.10962174088953602934493757712303036291","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606","series":"1.2.826.0.1.3680043.8.498.37479263340084972880605653317455129822","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94447683257072169359896104855962933413","series":"1.2.826.0.1.3680043.8.498.11990117080011704156574510720950387575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94590838906158914501563687877014778914","series":"1.2.826.0.1.3680043.8.498.57610799787229987091756599503541404463","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480","series":"1.2.826.0.1.3680043.8.498.26355503124652714463947064603323124796","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94635110520016467882028406590905441740","series":"1.2.826.0.1.3680043.8.498.92204023617779530779394059529057535142","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046","series":"1.2.826.0.1.3680043.8.498.51820535404927306879141660451243056389","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94747403767627336827372096100476526821","series":"1.2.826.0.1.3680043.8.498.77015805722767217224808101711142509525","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589","series":"1.2.826.0.1.3680043.8.498.67862843560108471383722589002740755959","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94822220672465914426995315295921716764","series":"1.2.826.0.1.3680043.8.498.10168500342173524713738592538404017596","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.77114982564360431519228347656326432073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224","series":"1.2.826.0.1.3680043.8.498.70305222328824208219313504669849994815","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745","series":"1.2.826.0.1.3680043.8.498.15886257898564256996799074797515746220","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95091624999382167943525022414631622532","series":"1.2.826.0.1.3680043.8.498.38846421575715644963676003321793594951","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95142886737239438773421088967802136318","series":"1.2.826.0.1.3680043.8.498.10254376833131008216082821079909208804","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95223549428749744711279985897241419512","series":"1.2.826.0.1.3680043.8.498.62018106179962997048412607339763055271","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139","series":"1.2.826.0.1.3680043.8.498.60630256675037930625027205978416899264","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95334149193711661424105715330320414755","series":"1.2.826.0.1.3680043.8.498.72854236988274744897819177960420304069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109","series":"1.2.826.0.1.3680043.8.498.39398029768976154656891133781358606403","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657","series":"1.2.826.0.1.3680043.8.498.15806564980594946070306982882392437570","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948","series":"1.2.826.0.1.3680043.8.498.61460747123326739291416140748773135970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375","series":"1.2.826.0.1.3680043.8.498.74429632408523784448028948666234615961","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863","series":"1.2.826.0.1.3680043.8.498.97812812884402305079043279910226378003","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698","series":"1.2.826.0.1.3680043.8.498.43733991359353661502249377209931293765","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180","series":"1.2.826.0.1.3680043.8.498.12360071282752382268132591041045036928","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460","series":"1.2.826.0.1.3680043.8.498.34503270402918880141824585180839038752","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603","series":"1.2.826.0.1.3680043.8.498.23521008752845241327110101328862833142","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95850844945129153291759207212815927078","series":"1.2.826.0.1.3680043.8.498.97791909868984001315293777437868532028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95912798959818676102590029534562878721","series":"1.2.826.0.1.3680043.8.498.99628903117667243851926758466601419829","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96051765028927322365509303222455708654","series":"1.2.826.0.1.3680043.8.498.69072787597021971910948534678732294870","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96094001324999448553107805455529187871","series":"1.2.826.0.1.3680043.8.498.11360092019960624975498341513354936369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96152894373838410779390571000523933753","series":"1.2.826.0.1.3680043.8.498.11728547891375690423110523782929069334","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96174494699876332875347380499122120925","series":"1.2.826.0.1.3680043.8.498.12663936315815924038741835073750054127","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96316229168827165128604998083731481211","series":"1.2.826.0.1.3680043.8.498.74967836545631035344732225210587926253","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420","series":"1.2.826.0.1.3680043.8.498.21306960528768727266075001710085401857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96559819562317669526803257835398470484","series":"1.2.826.0.1.3680043.8.498.76716772687579857827354449790059473952","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96562865861483086790615358719391771217","series":"1.2.826.0.1.3680043.8.498.10997803906020837009354696973671764770","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083","series":"1.2.826.0.1.3680043.8.498.79818117085214291402480442516493655093","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96657465815875587552665111166265086308","series":"1.2.826.0.1.3680043.8.498.88759047486975239517379387765045225832","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926","series":"1.2.826.0.1.3680043.8.498.18679253671987294334054268168630833903","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96943946564115188893954628924857155670","series":"1.2.826.0.1.3680043.8.498.76439588221302155540685234775054864018","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96947237995559790704304892745226362883","series":"1.2.826.0.1.3680043.8.498.30694089578625983397169606170273856477","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97004483591331502283638578516018997228","series":"1.2.826.0.1.3680043.8.498.11643074837097636920077978926705349108","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644","series":"1.2.826.0.1.3680043.8.498.63461351146266629916283057510983268070","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97107472223824971578324157203273471706","series":"1.2.826.0.1.3680043.8.498.11639334579440780459126542458414478077","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97220604868276521004168045790587997687","series":"1.2.826.0.1.3680043.8.498.48517557292178913030345410410288586579","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.90760449851947234877589521762844690352","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442","series":"1.2.826.0.1.3680043.8.498.43704838744387350842651276103698765844","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97296323335289879687414244774227160450","series":"1.2.826.0.1.3680043.8.498.36443759257805254531315917290022426004","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97353980946516912216109392482235435606","series":"1.2.826.0.1.3680043.8.498.33669714375633181456907339344537519742","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97464332941382646555656038793589898504","series":"1.2.826.0.1.3680043.8.498.54167095945326954985687702914881517845","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.16758853006497098427910475062504821725","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.11580810161122214085034769567431382504","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97564564380731258764291146761742841527","series":"1.2.826.0.1.3680043.8.498.12744791085501808094965102812684486325","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97658595748192892503053963872409769829","series":"1.2.826.0.1.3680043.8.498.47492502737765690650677064767674071338","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97694479664965569386338738579573728752","series":"1.2.826.0.1.3680043.8.498.26845490368277485306507464450489178609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.16886945038251896783044681883815984886","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97797693721516872008302328638374252668","series":"1.2.826.0.1.3680043.8.498.50670549085628178969789482906646147097","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97909881548577311929907613359449243075","series":"1.2.826.0.1.3680043.8.498.41407243388912771592188512087823219817","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97964867333695657200856428936254426086","series":"1.2.826.0.1.3680043.8.498.95372782484166919528775455526099222446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642","series":"1.2.826.0.1.3680043.8.498.39314859438461316891208262414725875923","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265","series":"1.2.826.0.1.3680043.8.498.86140214084538574013302564382255720806","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974","series":"1.2.826.0.1.3680043.8.498.10787768731062469786059516313700264424","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98143909986671737130865120569673002676","series":"1.2.826.0.1.3680043.8.498.11459501436093556715025174296613259723","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98151095279541278201351784048265686573","series":"1.2.826.0.1.3680043.8.498.16766651870143895830562493512646279680","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938","series":"1.2.826.0.1.3680043.8.498.21369274140073378305196974113902999112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.18509622150647189320886581028259321892","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98351891935507349137982199658103205287","series":"1.2.826.0.1.3680043.8.498.40741919800880076403582100417271781261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104","series":"1.2.826.0.1.3680043.8.498.10985779645450902559535902062557054223","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070","series":"1.2.826.0.1.3680043.8.498.36512812443905505257768088695398333359","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98752901295449528283322564256444025625","series":"1.2.826.0.1.3680043.8.498.13001918693714788391370641863004653673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186","series":"1.2.826.0.1.3680043.8.498.26363771796184774509366499780335946951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068","series":"1.2.826.0.1.3680043.8.498.11299326677544333888730908551924094968","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98914528123216365374152816424181510781","series":"1.2.826.0.1.3680043.8.498.13932313721050455351919572169208281821","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874","series":"1.2.826.0.1.3680043.8.498.12844605358810064572404776946753896175","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99088247184917460283293617180083591028","series":"1.2.826.0.1.3680043.8.498.11255625476747910331487715520454187381","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99178010550062268668346041362629896201","series":"1.2.826.0.1.3680043.8.498.83149428097255426179313290938644812070","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99257151613846084492342083136797222759","series":"1.2.826.0.1.3680043.8.498.69575461522508118053259553840063544338","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99309981642816644502709148844743995902","series":"1.2.826.0.1.3680043.8.498.28386404633546706787232492629680540451","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.92741038539839230631942423900901734473","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280","series":"1.2.826.0.1.3680043.8.498.66890737973748962413552712352001094170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928","series":"1.2.826.0.1.3680043.8.498.37869939615099823324936153248261949855","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.43732551983597414830499856301126265854","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99635784824711754116112241806442235135","series":"1.2.826.0.1.3680043.8.498.42725121902512084429460282795346233409","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99710592666731203419714499168506345263","series":"1.2.826.0.1.3680043.8.498.11533272009065922721576775833542323837","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99726731706411379395165830480890405115","series":"1.2.826.0.1.3680043.8.498.36289774375213403898577317865791322559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99826037216341310731701527407057049820","series":"1.2.826.0.1.3680043.8.498.84613178692255469944477449496305216101","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99864345386635409953374818045984558970","series":"1.2.826.0.1.3680043.8.498.43363579967112312556839899968806909718","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290","series":"1.2.826.0.1.3680043.8.498.48816564121976694830776136711636835161","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","series":"1.2.826.0.1.3680043.8.498.54309001783901668483872810054787793416","plane":"Axial","fluid":1,"slot":"AX_FS"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1778
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
